# M25b · Variables latentes et ELBO — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/m25b_variables_latentes/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). Dans les parties A à D, donne les valeurs que rend le calcul, **sans les arrondir** : les décimales demandées sont celles de ta copie. « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/m25b_variables_latentes/03_notebook.ipynb`, créée par `python tools/start_chapter.py M25b`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | M25b.1, 2, 3, 6, 9, 10, 12, 13, 14, 23 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | M25b.20 | Les responsabilités d'un mélange de gaussiennes, calculées en log (le pas E) · `gmm_e_step` | 🔨 | ★★ | 30 |
|  | M25b.21 | Le pas M d'un mélange de gaussiennes : poids, moyennes et covariances pondérées, avec une régularisation · `gmm_m_step` | 🔨 | ★★ | 30 |
|  | M25b.22 | La log-vraisemblance d'un mélange de gaussiennes, et son ELBO pour des responsabilités quelconques · `gmm_log_likelihood`, `gmm_elbo` | 🔨 | ★★ | 25 |
|  | M25b.24 | `scipy.linalg.subspace_angles` et `np.linalg.eigh` : les angles principaux de M25b.10, et l'ACP probabiliste de mesures par leurs valeurs propres, contre la vraisemblance de `mvn_logpdf` | 📦 | ★ | 20 |
| B | M25b.25 | Prédire ce que devient la log-vraisemblance quand une composante centrée sur un point se resserre (σ_k = 10⁻¹, 10⁻³, 10⁻⁶), puis avec une régularisation, et mesurer | 🔮 | ★ | 15 |
|  | M25b.26 | L'EM sur des longueurs de poissons de trois cohortes : la log-vraisemblance et l'ELBO après chaque demi-pas, sans régularisation, plusieurs initialisations et des maxima locaux | 🔬 | ★★ | 25 |
|  | M25b.27 | Une gaussienne ajustée à un mélange bimodal par KL(p ‖ q) puis par KL(q ‖ p) : les deux courbes, et les deux minima de la KL inverse | 🎨 | ★ | 20 |
| C | M25b.28 | Chasse à l'erreur : des responsabilités sans log qui donnent 0/0, un pas M qui divise par n au lieu de N_k, et une ELBO sans l'entropie de q | 🐛 | ★ | 20 |
|  | M25b.29 | Un test de propriété : sur des mélanges tirés au hasard, ELBO ≤ log-vraisemblance, égalité après le pas E, et un EM sans régularisation (reg = 0) qui ne descend jamais (à une tolérance près) | 🛠️ | ★ | 15 |
| D | M25b.30 | Défi : retrouver trois populations de mesures en dimension 2 par l'EM, sous un seuil de log-vraisemblance moyenne sur des mesures de test | 🏆 | ★★★ | 25 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M25b")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : un modèle à variable latente et son posterior, les moments d'un mélange, une itération de l'EM en log, l'ELBO d'un latent à deux états, l'ACP probabiliste, des angles principaux, la dérivée d'une espérance par le chemin, les deux sens de la KL, un code stochastique et son information mutuelle, et un pont vers les tableaux en lignes.
- Écrire les quatre fonctions de `mymath/latent.py` (`gmm_e_step`, `gmm_m_step`, `gmm_log_likelihood`, `gmm_elbo`) et les tester.
- Vérifier des angles principaux et ajuster une ACP probabiliste avec SciPy, prédire et mesurer une vraisemblance qui n'est pas bornée, suivre l'EM demi-pas par demi-pas, dessiner les deux sens de la KL, corriger trois erreurs classiques, écrire un test de propriété, et relever un défi.

**Rappel express.** $p(\mathbf{x}) = \sum_k\pi_k\,\mathcal{N}(\mathbf{x} \mid \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)$ ; $a_{ik} = \ln\pi_k + \ln\mathcal{N}(\mathbf{x}_i \mid \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)$, $\ln p(\mathbf{x}_i) = \operatorname{LSE}_j\,a_{ij}$, $\ln r_{ik} = a_{ik} - \operatorname{LSE}_j\,a_{ij}$, avec $\operatorname{LSE}_j\,a_{ij} = m + \ln\sum_je^{a_{ij} - m}$, $m = \max_ja_{ij}$. Pas M : $N_k = \sum_ir_{ik}$, $\pi_k = N_k/n$, $\boldsymbol{\mu}_k = \frac{1}{N_k}\sum_ir_{ik}\mathbf{x}_i$, $\boldsymbol{\Sigma}_k = \frac{1}{N_k}\sum_ir_{ik}(\mathbf{x}_i - \boldsymbol{\mu}_k)(\mathbf{x}_i - \boldsymbol{\mu}_k)^\top$ (+ `reg`·I). $\operatorname{ELBO} = \sum_i\sum_kr_{ik}(a_{ik} - \ln r_{ik}) \le \ell$, égalité au posterior ; l'écart est $\sum_i\mathrm{KL}(q_i \,\|\, p(z \mid \mathbf{x}_i))$ ; sans régularisation, l'EM ne la fait jamais baisser. Moments d'un mélange : $\boldsymbol{\mu} = \sum_k\pi_k\boldsymbol{\mu}_k$, $\boldsymbol{\Sigma} = \sum_k\pi_k\boldsymbol{\Sigma}_k + \sum_k\pi_k(\boldsymbol{\mu}_k - \boldsymbol{\mu})(\boldsymbol{\mu}_k - \boldsymbol{\mu})^\top$. ACP probabiliste : $\mathbf{C} = \mathbf{W}\mathbf{W}^\top + \sigma^2\mathbf{I}$, $\mathbf{z} \mid \mathbf{x} \sim \mathcal{N}(\mathbf{M}^{-1}\mathbf{W}^\top(\mathbf{x} - \boldsymbol{\mu}), \sigma^2\mathbf{M}^{-1})$, $\hat\sigma^2$ = moyenne des valeurs propres écartées, $\lVert\hat{\mathbf{w}}\rVert^2 = \lambda_1 - \hat\sigma^2$ à $k = 1$. Angles principaux : $\cos\phi_i$ = valeurs singulières de $\mathbf{Q}_A^\top\mathbf{Q}_B$. $\mathrm{KL}(p \,\|\, q)$ apparie les moments, $\mathrm{KL}(q \,\|\, p)$ choisit un mode.

## Partie 0 · Vérifier tes exercices papier (✏️ M25b.1, M25b.2, M25b.3, M25b.6, M25b.9, M25b.10, M25b.12, M25b.13, M25b.14 ; 🔗 M25b.23)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md` (le 🔗 M25b.23 après le 🔨 M25b.22). Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.25`, arrondi comme le demande la question ;
- une liste `[0.2, -1]` ; une matrice comme la liste de ses **lignes**, `[[1, 0], [-2, 3]]` ;
- une fraction irréductible entre guillemets, `"3/7"`, quand la question le dit ;
- une valeur exacte entre guillemets, écrite avec des entiers, des fractions, `sqrt`, `pi`, `ln` et `exp`, comme `"ln(5/2)"` ;
- une **forme** comme un tuple, `(7, 2)`, ou comme un texte, `"(7, 2)"` ;
- `True` ou `False` pour un vrai ou faux.

Les réponses pas encore remplies affichent ⏳.

**Ex M25b.1 — Trois machines, des tiges de longueurs gaussiennes : tirage ancestral, vraisemblance marginale et posterior de la machine**

In [ ]:
# Ex M25b.1: the values you found on paper
answer_M25b_1a = ...  # a) a list of 2 numbers: the number of the machine, then the length (mm)
answer_M25b_1b = ...  # b) a number (4 decimals)
answer_M25b_1c = ...  # c) a list of 3 numbers (3 decimals)
answer_M25b_1d = ...  # d) an integer
answer_M25b_1e = ...  # e) True or False
answer_M25b_1f = ...  # f) a number (4 decimals)

for letter, answer in zip("abcdef", [answer_M25b_1a, answer_M25b_1b, answer_M25b_1c, answer_M25b_1d, answer_M25b_1e, answer_M25b_1f]):
    wbm.check(f"M25b.1{letter}", answer)

**Ex M25b.2 — Deux variétés de grains de café : moyenne et covariance du mélange, part entre variétés, corrélation du sac**

In [ ]:
# Ex M25b.2: the values you found on paper
answer_M25b_2a = ...  # a) a list of 2 numbers
answer_M25b_2b = ...  # b) a 2 × 2 matrix
answer_M25b_2c = ...  # c) a 2 × 2 matrix
answer_M25b_2d = ...  # d) a 2 × 2 matrix
answer_M25b_2e = ...  # e) an irreducible fraction, as text "p/q"
answer_M25b_2f = ...  # f) a number (4 decimals)
answer_M25b_2g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_M25b_2a, answer_M25b_2b, answer_M25b_2c, answer_M25b_2d, answer_M25b_2e, answer_M25b_2f, answer_M25b_2g]):
    wbm.check(f"M25b.2{letter}", answer)

**Ex M25b.3 — Deux séries de résistances : une itération de l'EM en log, et la log-vraisemblance qui monte**

In [ ]:
# Ex M25b.3: the values you found on paper
answer_M25b_3a = ...  # a) a list of 4 numbers (4 decimals)
answer_M25b_3b = ...  # b) a list of 2 numbers (4 decimals)
answer_M25b_3c = ...  # c) a list of 2 numbers (3 decimals)
answer_M25b_3d = ...  # d) a list of 2 numbers (3 decimals)
answer_M25b_3e = ...  # e) a number (4 decimals)
answer_M25b_3f = ...  # f) a number (4 decimals)
answer_M25b_3g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_M25b_3a, answer_M25b_3b, answer_M25b_3c, answer_M25b_3d, answer_M25b_3e, answer_M25b_3f, answer_M25b_3g]):
    wbm.check(f"M25b.3{letter}", answer)

**Ex M25b.6 — Une sonde de qualité de l'eau : ln p(x), l'ELBO pour trois lois q et l'écart égal à la KL**

In [ ]:
# Ex M25b.6: the values you found on paper
answer_M25b_6a = ...  # a) a number (4 decimals)
answer_M25b_6b = ...  # b) a number (3 decimals)
answer_M25b_6c = ...  # c) a number (4 decimals)
answer_M25b_6d = ...  # d) a number (4 decimals)
answer_M25b_6e = ...  # e) a number (4 decimals)
answer_M25b_6f = ...  # f) a number (4 decimals)
answer_M25b_6g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_M25b_6a, answer_M25b_6b, answer_M25b_6c, answer_M25b_6d, answer_M25b_6e, answer_M25b_6f, answer_M25b_6g]):
    wbm.check(f"M25b.6{letter}", answer)

**Ex M25b.9 — Trois épreuves d'un concours, une aptitude cachée : l'ACP probabiliste à un facteur**

In [ ]:
# Ex M25b.9: the values you found on paper
answer_M25b_9a = ...  # a) a 3 × 3 matrix
answer_M25b_9b = ...  # b) a number
answer_M25b_9c = ...  # c) a list of 3 numbers (4 decimals)
answer_M25b_9d = ...  # d) a list of 2 numbers (4 decimals)
answer_M25b_9e = ...  # e) True or False
answer_M25b_9f = ...  # f) a number
answer_M25b_9g = ...  # g) a number
answer_M25b_9h = ...  # h) a list of 2 numbers
answer_M25b_9i = ...  # i) True or False

for letter, answer in zip("abcdefghi", [answer_M25b_9a, answer_M25b_9b, answer_M25b_9c, answer_M25b_9d, answer_M25b_9e, answer_M25b_9f, answer_M25b_9g, answer_M25b_9h, answer_M25b_9i]):
    wbm.check(f"M25b.9{letter}", answer)

**Ex M25b.10 — Les angles principaux entre deux plans de ℝ⁴ et entre deux droites de ℝ³**

In [ ]:
# Ex M25b.10: the values you found on paper
answer_M25b_10a = ...  # a) a 4 × 2 matrix (4 decimals)
answer_M25b_10b = ...  # b) a 4 × 2 matrix (4 decimals)
answer_M25b_10c = ...  # c) a 2 × 2 matrix (4 decimals)
answer_M25b_10d = ...  # d) a list of 2 numbers (4 decimals)
answer_M25b_10e = ...  # e) a list of 2 numbers (2 decimals)
answer_M25b_10f = ...  # f) a number (2 decimals)
answer_M25b_10g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_M25b_10a, answer_M25b_10b, answer_M25b_10c, answer_M25b_10d, answer_M25b_10e, answer_M25b_10f, answer_M25b_10g]):
    wbm.check(f"M25b.10{letter}", answer)

**Ex M25b.12 — Le coût d'un stock face à une demande gaussienne : E[(D − c)⁺] et ses dérivées par le chemin**

In [ ]:
# Ex M25b.12: the values you found on paper
answer_M25b_12a = ...  # a) a number (2 decimals)
answer_M25b_12b = ...  # b) a number (4 decimals)
answer_M25b_12c = ...  # c) a number (4 decimals)
answer_M25b_12d = ...  # d) a list of 2 numbers
answer_M25b_12e = ...  # e) a list of 2 numbers
answer_M25b_12f = ...  # f) a number (4 decimals)
answer_M25b_12g = ...  # g) a number (4 decimals)

for letter, answer in zip("abcdefg", [answer_M25b_12a, answer_M25b_12b, answer_M25b_12c, answer_M25b_12d, answer_M25b_12e, answer_M25b_12f, answer_M25b_12g]):
    wbm.check(f"M25b.12{letter}", answer)

**Ex M25b.13 — Une seule gaussienne pour les temps de trajet par deux itinéraires : les deux sens de la KL**

In [ ]:
# Ex M25b.13: the values you found on paper
answer_M25b_13a = ...  # a) a list of 2 numbers
answer_M25b_13b = ...  # b) a number (2 decimals)
answer_M25b_13c = ...  # c) an exact value, as text, with sqrt, pi, ln, exp and fractions
answer_M25b_13d = ...  # d) an exact value, as text, with sqrt, pi, ln, exp and fractions
answer_M25b_13e = ...  # e) an integer
answer_M25b_13f = ...  # f) a number (4 decimals)
answer_M25b_13g = ...  # g) a number (4 decimals)
answer_M25b_13h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M25b_13a, answer_M25b_13b, answer_M25b_13c, answer_M25b_13d, answer_M25b_13e, answer_M25b_13f, answer_M25b_13g, answer_M25b_13h]):
    wbm.check(f"M25b.13{letter}", answer)

**Ex M25b.14 — Un code stochastique d'une source à trois symboles : posterior agrégé, information mutuelle, code effondré**

In [ ]:
# Ex M25b.14: the values you found on paper
answer_M25b_14a = ...  # a) a list of 2 numbers (4 decimals)
answer_M25b_14b = ...  # b) a list of 3 numbers (4 decimals)
answer_M25b_14c = ...  # c) a number (4 decimals)
answer_M25b_14d = ...  # d) a number (4 decimals)
answer_M25b_14e = ...  # e) a number (4 decimals)
answer_M25b_14f = ...  # f) True or False
answer_M25b_14g = ...  # g) a list of 2 numbers
answer_M25b_14h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M25b_14a, answer_M25b_14b, answer_M25b_14c, answer_M25b_14d, answer_M25b_14e, answer_M25b_14f, answer_M25b_14g, answer_M25b_14h]):
    wbm.check(f"M25b.14{letter}", answer)

**Ex M25b.23 — Pont : l'ELBO de mesures rangées en lignes, « reconstruction moins KL au prior », en nats par mesure**

In [ ]:
# Ex M25b.23: the values you found on paper
answer_M25b_23a = ...  # a) a shape: a tuple (7, 2) or a text "(7, 2)"
answer_M25b_23b = ...  # b) a number (4 decimals)
answer_M25b_23c = ...  # c) a number (4 decimals)
answer_M25b_23d = ...  # d) a number (4 decimals)
answer_M25b_23e = ...  # e) a list of 3 numbers (3 decimals)
answer_M25b_23f = ...  # f) a list of 2 numbers (4 decimals)
answer_M25b_23g = ...  # g) a number (4 decimals)
answer_M25b_23h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M25b_23a, answer_M25b_23b, answer_M25b_23c, answer_M25b_23d, answer_M25b_23e, answer_M25b_23f, answer_M25b_23g, answer_M25b_23h]):
    wbm.check(f"M25b.23{letter}", answer)

## Partie A · Ta librairie : le pas E, le pas M, la log-vraisemblance et l'ELBO ; puis SciPy

Fiche §3 à §6. Les quatre fonctions de `mymath/latent.py`, leurs tests, et un usage de chacune ; puis `scipy.linalg.subspace_angles` et `np.linalg.eigh` pour les angles principaux et l'ACP probabiliste.

In [ ]:
# Tools of the notebook (parts A to D)
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
from scipy import linalg, optimize, special, stats

from wbm.errors import MymathMissingError
from wbm.nbtools import StopCheck, checked, drawn, filled, fr, own_code, pair, run_pytest_group, same, verdict

TEST_M25B = "tests/test_m25b_latent.py"            # the tests of mymath/latent.py
REF = wbm.load_mymath("ref", alias="mymath_ref")   # the reference: the data cells and the checks never use your code
ref_gauss, ref_latent = REF.gaussian, REF.latent
REF_LATENT = {f"ref_latent.{name}": getattr(ref_latent, name)    # a check stops if your code calls one of them
              for name in ("gmm_e_step", "gmm_m_step", "gmm_log_likelihood", "gmm_elbo")}


def shown(value, digits=4):
    """A short text for a result of the learner's code (the check messages): numbers rounded, as lists."""
    if value is None:
        return "None"
    try:
        if isinstance(value, tuple):
            return "(" + ", ".join(str(np.round(np.asarray(v, dtype=float), digits).tolist()) for v in value) + ")"
        return str(np.round(np.asarray(value, dtype=float), digits).tolist())
    except (TypeError, ValueError):
        return repr(value)[:120]


def reporter(results, stop=True):
    """A function report(ok, success, failure) that prints one ✅ or ❌ line, keeps the verdict in results, and (stop)
    ends the check at the first ❌ (StopCheck, caught by checked)."""
    def report(ok, success, failure):
        results.append(bool(ok))
        print(("✅ " + success) if ok else ("❌ " + failure))
        if not ok and stop:
            raise StopCheck
    return report


def line_points(line):
    """The (x, y) points of a line of matplotlib, as two float arrays."""
    return np.asarray(line.get_xdata(), dtype=float), np.asarray(line.get_ydata(), dtype=float)


def through(ax, xs, ys, rtol=1e-9, atol=1e-12):
    """True when one line of ax passes by all the points (xs, ys), in this order or in the reverse order."""
    xs, ys = np.asarray(xs, dtype=float), np.asarray(ys, dtype=float)
    for line in ax.lines:
        x, y = line_points(line)
        for a, b in ((xs, ys), (xs[::-1], ys[::-1])):
            if (x.shape == a.shape and y.shape == b.shape and np.allclose(x, a, rtol=rtol, atol=atol)
                    and np.allclose(y, b, rtol=rtol, atol=atol)):
                return True
    return False


def has_points(ax, points):
    """True when every (x, y) of points is a point of a line (markers) or of a scatter of ax."""
    found = []
    for line in ax.lines:
        x, y = line_points(line)
        found.append(np.column_stack([x, y]))
    for collection in ax.collections:
        offsets = np.asarray(collection.get_offsets(), dtype=float)
        if offsets.ndim == 2 and offsets.shape[1] == 2:
            found.append(offsets)
    if not found:
        return False
    found = np.vstack(found)
    points = np.asarray(points, dtype=float)
    return all(bool(np.any(np.all(np.abs(found - p) < 1e-9, axis=1))) for p in points)


def matches(got, want, rtol=1e-9, atol=1e-12):
    """True when got is an array of numbers of the shape of want and equal to it up to rounding (no error for an odd
    got): to recognize a classic wrong result before the comparison by same."""
    try:
        arr = np.asarray(got, dtype=float)
    except (TypeError, ValueError):
        return False
    want = np.asarray(want, dtype=float)
    return arr.shape == want.shape and bool(np.allclose(arr, want, rtol=rtol, atol=atol))


def scipy_log_joint(X, weights, means, covs):
    """a[i, k] = ln π_k + ln N(x_i | μ_k, Σ_k) by scipy.stats.multivariate_normal: the oracle of the checks."""
    X = np.asarray(X, dtype=float)
    columns = [np.reshape(stats.multivariate_normal(means[k], covs[k]).logpdf(X), (len(X),)) for k in range(len(weights))]
    return np.column_stack(columns) + np.log(weights)


def scipy_log_likelihood(X, weights, means, covs):
    """Σ_i logsumexp_k a[i, k], by SciPy."""
    return float(np.sum(special.logsumexp(scipy_log_joint(X, weights, means, covs), axis=1)))


def ellipse_drawn(ax, mean, cov, level):
    """True when a closed curve of ax lies on the ellipse of N(mean, cov) that contains the mass level (Δ² = c²)."""
    mean, L = np.asarray(mean, dtype=float), np.linalg.cholesky(np.asarray(cov, dtype=float))
    c2 = -2 * math.log(1 - level)
    for line in ax.lines:
        x, y = line_points(line)
        if x.size < 20:
            continue
        white = np.linalg.solve(L, np.vstack([x - mean[0], y - mean[1]])).T
        if np.allclose(np.sum(white ** 2, axis=1), c2, rtol=1e-6, atol=1e-9):
            angles = np.sort(np.arctan2(white[:, 1], white[:, 0]))
            if np.diff(np.concatenate([angles, [angles[0] + 2 * np.pi]])).max() < 0.5:
                return True
    return False

### Ex M25b.20 — Les responsabilités d'un mélange de gaussiennes, calculées en log (le pas E) · `gmm_e_step` 🔨 ★★ ⏱️ 30 min
**Objectif :** programmer le pas E d'un mélange de gaussiennes en log, puis voir que loin de toutes les composantes il ne donne jamais 0/0.  
**Prérequis :** fiche §3 ; M05a, M25a ; Ex M25b.1 et M25b.3 · **mymath :** `latent.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en M25a). Ouvre ta copie `mon_travail/mymath/latent.py`, lis la docstring du module puis celle de la fonction, remplace chaque `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. Dans le notebook, appelle tes fonctions par `mymath.latent.<nom>` (par exemple `mymath.latent.gmm_e_step(X, weights, means, covs)`), et celles de M25a par `mymath.gaussian.<nom>` (`mymath.gaussian.mvn_logpdf`, `mymath.gaussian.sample_mvn`, `mymath.gaussian.condition_gaussian`), ou donne-leur un nom court (`gmm_e_step = mymath.latent.gmm_e_step`) dans une cellule exécutée après le rechargement. ⚠️ Le squelette n'importe rien : ajoute en tête du fichier `import numpy as np` et `from .gaussian import mvn_logpdf`, un import relatif de ta log-densité de M25a. **Si ton `gaussian.py` contient encore le stub de `mvn_logpdf`, les tests de `latent` échouent** (« ⏳ `mvn_logpdf()` n'est pas encore écrite ») : termine d'abord le 🔨 M25a.20 ; sans fichier `gaussian.py`, celui de la référence est utilisé (une ligne ℹ️ le dit). NumPy est permis, mais ni `np.cov` ni `np.average`, de `np.linalg` seulement `cholesky` et `solve` (et `np.linalg.LinAlgError`), aucune fonction de SciPy ni de scikit-learn : les tests comparent tes résultats aux leurs, et un test lit ton module. Le log-sum-exp de chaque ligne d'un tableau $(n, K)$ s'écrit en NumPy, en retirant le maximum de la ligne (celui d'`approx`, M05a, ne prend qu'une séquence). Les données sont les **lignes** d'un tableau $(n, d)$ (des mesures en dimension 1 forment une colonne $(n, 1)$) ; les poids ont la forme $(K,)$, les moyennes $(K, d)$, les covariances $(K, d, d)$ et les responsabilités $(n, K)$. Les nombres se rendent en `float` de Python, les tableaux en tableaux NumPy de flottants ; ne modifie jamais les tableaux reçus. Hors de `mymath`, le notebook peut se servir de tout NumPy et de SciPy.

Écris `gmm_e_step(X, weights, means, covs)` (fiche §3), en suivant sa docstring : le tableau $a_{ik} = \ln\pi_k + \ln\mathcal{N}(\mathbf{x}_i \mid \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)$ de forme $(n, K)$, une colonne de `mvn_logpdf(X, means[k], covs[k])` par composante, auquel on retire le log-sum-exp de chaque **ligne**, puis l'exponentielle. Un tableau $(n, K)$ dont chaque ligne a pour somme 1 ; `ValueError` pour les formes, les poids et les covariances que refuse la docstring.

**L'usage.** La cellule fournie donne les quatre résistances de M25b.3 (`X_RES_20`, de forme $(4, 1)$) avec les paramètres de départ (`WEIGHTS_RES_20`, `MEANS_RES_20`, `COVS_RES_20` de forme $(2, 1, 1)$), et la tige de 98,2 mm de M25b.1 (`X_ROD_20`) avec ses trois machines (`WEIGHTS_ROD_20`, `MEANS_ROD_20`, et les **écarts-types** `SDS_ROD_20`). Avec ta fonction :
- a) les responsabilités $r_{i1}$ de la série 1 pour les quatre résistances (une liste de 4 nombres), à comparer à ta réponse a) de M25b.3 ;
- b) on ajoute en cinquième ligne une résistance de 120 Ω égarée dans le bac : sa ligne de responsabilités, et vrai ou faux, le tableau $(5, 2)$ ne contient aucun `nan` (`np.isnan(...).any()`) (une liste : les deux nombres, puis le booléen) ;
- c) la ligne de la tige de 98,2 mm avec les trois machines (`covs` de forme $(3, 1, 1)$ : les variances), à comparer à ta réponse c) de M25b.1 (une liste de 3 nombres).

Ce qu'il faut observer : loin de toutes les composantes, c'est la plus large qui prend toute la responsabilité, et le calcul en log ne donne jamais $0/0$. Dans ta copie : pourquoi la responsabilité de la série 1 pour 52,1 Ω est-elle presque nulle ?

In [ ]:
X_RES_20 = np.array([[46.0], [48.0], [49.4], [52.1]])    # the four resistors of M25b.3 (Ω): an array (4, 1)
WEIGHTS_RES_20 = np.array([0.45, 0.55])                    # the starting parameters of M25b.3
MEANS_RES_20 = np.array([[47.0], [51.0]])                  # (K, d) = (2, 1)
COVS_RES_20 = np.array([[[1.0]], [[2.5]]])                 # (K, d, d) = (2, 1, 1): the variances (Ω²)
X_ROD_20 = np.array([[98.2]])                              # the rod of M25b.1 (mm)
WEIGHTS_ROD_20 = np.array([0.5, 0.3, 0.2])                 # the three machines
MEANS_ROD_20 = np.array([[100.0], [102.0], [97.0]])
SDS_ROD_20 = np.array([1.0, 1.0, 2.0])                     # their STANDARD DEVIATIONS (covs takes the variances)

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M25b")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_M25B, "test_gmm_e_step_ or test_module_uses_")

**L'usage** : les résistances, une résistance égarée, et la tige de M25b.1.

In [ ]:
answer_M25b_20a = answer_M25b_20b = answer_M25b_20c = ...
with wbm.attempt("M25b.20"):          # if you call gmm_e_step here: ⏳ while it is not written
    answer_M25b_20a = ...  # a) a list of 4 numbers, as computed: the responsibilities r_i1 (series 1) of the four resistors
    answer_M25b_20b = ...  # b) a list of 3: the row of the 120 Ω resistor (2 numbers, as computed), then True or False: your (5, 2) array holds no nan
    answer_M25b_20c = ...  # c) a list of 3 numbers, as computed: the row of the rod of 98.2 mm (three machines)

In [ ]:
for letter, answer in zip("abc", [answer_M25b_20a, answer_M25b_20b, answer_M25b_20c]):
    wbm.check(f"M25b.20{letter}", answer, computed=True)

### Ex M25b.21 — Le pas M d'un mélange de gaussiennes : poids, moyennes et covariances pondérées, avec une régularisation · `gmm_m_step` 🔨 ★★ ⏱️ 30 min
**Objectif :** programmer le pas M d'un mélange de gaussiennes, le maximum de vraisemblance pondéré de M25a, puis voir une composante s'effondrer sur une seule donnée.  
**Prérequis :** fiche §4 ; M25a ; Ex M25b.3 et M25b.20 · **mymath :** `latent.py` · **Parcours :** C, E

> Même mode d'emploi qu'en M25b.20 : écris la fonction dans ta copie `mon_travail/mymath/latent.py`, **enregistre**, puis exécute les cellules ci-dessous ; la première recharge ta librairie.

Écris `gmm_m_step(X, resp, reg=0.0)` (fiche §4), en suivant sa docstring : $N_k = \sum_i r_{ik}$, les poids $N_k/n$, les moyennes pondérées, puis les covariances pondérées autour des **nouvelles** moyennes, divisées par $N_k$, plus `reg` sur la diagonale. Le triplet `(weights, means, covs)` ; `ValueError` pour les formes, les responsabilités et les `reg` que refuse la docstring (une colonne nulle, une composante vide, est refusée).

**L'usage.** Avec les quatre résistances de M25b.20 (`X_RES_20`) et ta fonction :
- a) le pas M à partir de tes responsabilités de M25b.20 a) (le tableau $(4, 2)$ entier) : les poids, les moyennes, puis les variances (une liste de 6 nombres), à comparer à tes réponses b), c) et d) de M25b.3 ;
- b) des responsabilités dures, chaque résistance allant à la valeur nominale la plus proche (`HARD_21`) : les poids, les moyennes et les variances, et vrai ou faux, ce sont la proportion, `np.mean` et `np.var` de chaque groupe (une liste de 7 : les 6 nombres, puis le booléen) ;
- c) vrai ou faux : avec `reg=0.01`, les covariances dépassent celles de a) d'exactement 0,01 (à $10^{-12}$ près) ;
- d) avec `ALONE_21`, la composante 1 n'a qu'une résistance : sa variance rendue (sans régularisation), et vrai ou faux, `gmm_e_step` refuse ces paramètres par une `ValueError` (une liste : le nombre, puis le booléen ; attrape l'erreur par `try` / `except ValueError`).

Ce qu'il faut observer : on divise par $N_k$ ; une composante portée par une seule donnée a une variance nulle, la singularité de la fiche §3, et `reg` la garde définie positive.

In [ ]:
HARD_21 = np.array([[1, 0], [1, 0], [0, 1], [0, 1]])       # b) each resistor to the nearest nominal value
ALONE_21 = np.array([[1, 0], [0, 1], [0, 1], [0, 1]])      # d) component 1 carried by one resistor only

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M25b")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_M25B, "test_gmm_m_step_")

**L'usage** : un pas M sur les résistances, des responsabilités dures, `reg`, et une composante seule.

In [ ]:
answer_M25b_21a = answer_M25b_21b = answer_M25b_21c = answer_M25b_21d = ...
with wbm.attempt("M25b.21"):          # if you call gmm_m_step here: ⏳ while it is not written
    answer_M25b_21a = ...  # a) a list of 6 numbers, as computed: the weights, the means, then the variances after the M step from your (4, 2) responsibilities of M25b.20 a)
    answer_M25b_21b = ...  # b) a list of 7: the weights, the means, the variances for HARD_21 (6 numbers, as computed), then True or False: they are the proportion, np.mean and np.var of each group
    answer_M25b_21c = ...  # c) True or False: with reg=0.01, the covariances exceed those of a) by exactly 0.01 (to 1e-12)
    answer_M25b_21d = ...  # d) a list of 2: the variance of component 1 for ALONE_21 (as computed, reg = 0), then True or False: gmm_e_step refuses these parameters with a ValueError

In [ ]:
for letter, answer in zip("abcd", [answer_M25b_21a, answer_M25b_21b, answer_M25b_21c, answer_M25b_21d]):
    wbm.check(f"M25b.21{letter}", answer, computed=True)

### Ex M25b.22 — La log-vraisemblance d'un mélange de gaussiennes, et son ELBO pour des responsabilités quelconques · `gmm_log_likelihood`, `gmm_elbo` 🔨 ★★ ⏱️ 25 min
**Objectif :** programmer la log-vraisemblance d'un mélange et son ELBO pour des lois q quelconques, puis voir l'ELBO sous la log-vraisemblance, qu'elle touche au posterior.  
**Prérequis :** fiche §3 et §5 ; M06 ; Ex M25b.3, M25b.23 et M25b.21 · **mymath :** `latent.py` · **Parcours :** C, E

> Même mode d'emploi qu'en M25b.20 : écris la fonction dans ta copie `mon_travail/mymath/latent.py`, **enregistre**, puis exécute les cellules ci-dessous ; la première recharge ta librairie.

Écris `gmm_log_likelihood(X, weights, means, covs)` (fiche §3) : le même tableau $a$ qu'au pas E, le log-sum-exp de chaque ligne, puis la somme, un `float`. Écris ensuite `gmm_elbo(X, resp, weights, means, covs)` (fiche §5) : l'ELBO pour des lois $q_i$ quelconques (les lignes de `resp`), $\sum_i\sum_kr_{ik}(a_{ik} - \ln r_{ik})$, avec $0\ln 0 = 0$ (sans jamais calculer $\ln 0$), un `float`. `ValueError` pour ce que refusent les docstrings (pour `gmm_elbo`, une colonne nulle de `resp` est permise).

**L'usage.** La cellule fournie donne les trois pièces du 🔗 M25b.23 (`X_PIECES_22`, de forme $(3, 1)$, `WEIGHTS_PIECES_22`, `MEANS_PIECES_22`, `COVS_PIECES_22`) et le tableau $\mathbf{R}$ (`R_PIECES_22`). Avec tes fonctions :
- a) la log-vraisemblance des quatre résistances (`X_RES_20`) avant et après le pas M de M25b.21 a) (une liste de 2 nombres), à comparer à tes réponses e) et f) de M25b.3 ;
- b) pour les trois pièces : l'ELBO par mesure avec $\mathbf{R}$, puis avec les responsabilités de `gmm_e_step`, à comparer à tes réponses d) et g) de M25b.23, et vrai ou faux, la seconde égale `gmm_log_likelihood(X_PIECES_22, …) / 3` à $10^{-12}$ près (une liste : les deux nombres, puis le booléen) ;
- c) la log-vraisemblance des cinq résistances (avec celle de 120 Ω, paramètres de départ de M25b.3), et vrai ou faux, elle est finie (`np.isfinite`) (une liste : le nombre, puis le booléen) ;
- d) l'écart entre la log-vraisemblance des quatre résistances (paramètres de départ de M25b.3) et l'ELBO des responsabilités dures `HARD_21` de M25b.21 b), avec les mêmes paramètres : $\ell - \operatorname{ELBO}$ (un nombre).

Ce qu'il faut observer : l'ELBO reste sous la log-vraisemblance et la touche au posterior ; l'écart est une somme de KL. Dans ta copie : pourquoi l'écart de d) vaut-il la somme des $-\ln r_{ik}$ des composantes choisies ?

In [ ]:
X_PIECES_22 = np.array([[9.2], [11.6], [13.4]])           # the three pieces of M25b.23 (g)
WEIGHTS_PIECES_22 = np.array([0.75, 0.25])                 # two machines: N(10, 1) and N(14, 2.25)
MEANS_PIECES_22 = np.array([[10.0], [14.0]])
COVS_PIECES_22 = np.array([[[1.0]], [[2.25]]])
R_PIECES_22 = np.array([[0.85, 0.15], [0.5, 0.5], [0.3, 0.7]])   # the laws q_i of M25b.23, one row per piece

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M25b")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_M25B, "test_gmm_log_likelihood_ or test_gmm_elbo_")

**L'usage** : la log-vraisemblance qui monte, l'ELBO des pièces, une mesure lointaine, et un écart.

In [ ]:
answer_M25b_22a = answer_M25b_22b = answer_M25b_22c = answer_M25b_22d = ...
with wbm.attempt("M25b.22"):          # if you call your functions here: ⏳ while they are not written
    answer_M25b_22a = ...  # a) a list of 2 numbers, as computed: the log-likelihood of the four resistors before, then after the M step of M25b.21 a)
    answer_M25b_22b = ...  # b) a list of 3: the ELBO per measurement of the three pieces with R_PIECES_22, then with the responsibilities of gmm_e_step (2 numbers, as computed), then True or False: the second equals gmm_log_likelihood(...) / 3 to 1e-12
    answer_M25b_22c = ...  # c) a list of 2: the log-likelihood of the five resistors, 120 Ω included (as computed), then True or False: it is finite
    answer_M25b_22d = ...  # d) a number, as computed: the log-likelihood of the four resistors minus the ELBO of HARD_21 (starting parameters of M25b.3)

In [ ]:
for letter, answer in zip("abcd", [answer_M25b_22a, answer_M25b_22b, answer_M25b_22c, answer_M25b_22d]):
    wbm.check(f"M25b.22{letter}", answer, computed=True)

### Ex M25b.24 — `scipy.linalg.subspace_angles` et `np.linalg.eigh` : les angles principaux de M25b.10, et l'ACP probabiliste de mesures par leurs valeurs propres, contre la vraisemblance de `mvn_logpdf` 📦 ★ ⏱️ 20 min
**Objectif :** vérifier des angles principaux avec SciPy, puis ajuster une ACP probabiliste par les valeurs propres d'une covariance et comparer des vraisemblances.  
**Prérequis :** fiche §6 ; M12d, M25a ; Ex M25b.9 et M25b.10 · **Parcours :** C, E

La cellule fournie donne les deux plans de M25b.10 par leurs bases **non orthonormées** (`A_24` a pour colonnes $\mathbf{a}_1$ et $\mathbf{a}_2$, `B_24` a pour colonnes $\mathbf{b}_1$ et $\mathbf{b}_2$), les deux droites (`U_24`, `V_24`, des colonnes), et le modèle de M25b.9 (`W_24`, `MU_24`, `NOISE_24`, la covariance marginale `C_24`).
- a) Lis la documentation de `scipy.linalg.subspace_angles` (`help(linalg.subspace_angles)` ou la page en ligne) : dans quelle unité et dans quel ordre rend-elle les angles ? (A) radians, croissants ; (B) radians, décroissants ; (C) degrés, croissants ; (D) degrés, décroissants (une lettre).
- b) Applique-la aux deux plans (`angles_24`, le tableau rendu tel quel), puis aux deux droites : vrai ou faux, `np.cos` des angles rendus, rangés comme tes valeurs singulières de M25b.10 d), leur est égal à $10^{-12}$ près (et, pour les droites, au cosinus de ta réponse f) de M25b.10).
- c) On tire 500 candidats du modèle de M25b.9 : `X = sample_mvn(MU_24, C_24, 500, seed=6054)` (ta `mymath.gaussian.sample_mvn`). Avec la covariance du maximum de vraisemblance $\hat{\boldsymbol{\Sigma}}$ (`np.cov(X, rowvar=False, bias=True)`) et ses valeurs et vecteurs propres par `np.linalg.eigh` (rangés par ordre **croissant** : retourne-les), ajuste l'ACP probabiliste à $k = 1$ par Tipping et Bishop (fiche §6) : $\hat\sigma^2$ et $\lVert\hat{\mathbf{w}}\rVert^2$, puis la log-vraisemblance moyenne par candidat sous $\mathcal{N}(\bar{\mathbf{x}}, \hat{\mathbf{w}}\hat{\mathbf{w}}^\top + \hat\sigma^2\mathbf{I})$, par ta `mvn_logpdf` ; et trois vrai ou faux : elle est plus petite que sous $\mathcal{N}(\bar{\mathbf{x}}, \hat{\boldsymbol{\Sigma}})$ ; elle baisse si l'on remplace $\hat\sigma^2$ par $1{,}1\,\hat\sigma^2$ (même $\hat{\mathbf{w}}$) ; elle est plus grande que sous les vrais paramètres $\mathcal{N}(\boldsymbol{\mu}, \mathbf{C})$ (une liste de 6 : les trois nombres, puis les trois booléens ; 2, 2 et 4 décimales dans ta copie).
- d) Le posterior du candidat noté $(15, 12, 15)$ (`CANDIDATE_24`), dans le vrai modèle, comme loi conditionnelle de la loi jointe de $(z, \mathbf{x})$ en dimension 4, par ta `condition_gaussian` : moyenne $(0, \boldsymbol{\mu})$, covariance par blocs $\begin{pmatrix} 1 & \mathbf{w}^\top \\ \mathbf{w} & \mathbf{C} \end{pmatrix}$, indices `[1, 2, 3]` ; la moyenne et la variance a posteriori (une liste de 2 nombres ; 4 décimales dans ta copie), à comparer à ta réponse d) de M25b.9.

La vérification compare aussi `angles_24` au tableau que rend SciPy. Ce qu'il faut observer : la formule de Tipping et Bishop donne le maximum de vraisemblance parmi les modèles à un facteur ; le modèle complet a plus de paramètres et fait mieux sur ses propres données, comme l'estimation fait mieux que la vérité.

In [ ]:
A_24 = np.array([[1.0, 2.0], [2.0, 1.0], [0.0, 0.0], [0.0, 0.0]])        # columns a1, a2: the plane A of M25b.10
B_24 = np.array([[12.0, 12.0], [0.0, 7.0], [5.0, 5.0], [0.0, 24.0]])     # columns b1, b2: the plane B
U_24, V_24 = np.array([[1.0], [4.0], [8.0]]), np.array([[-4.0], [-8.0], [1.0]])   # the two lines of R³ (columns)
W_24 = np.array([2.0, 1.0, 2.0])                           # the model of M25b.9: x = w z + μ + ε
MU_24 = np.array([12.0, 11.0, 13.0])
NOISE_24 = 3.0                                             # σ²
C_24 = np.outer(W_24, W_24) + NOISE_24 * np.eye(3)         # the marginal covariance w wᵀ + σ² I
CANDIDATE_24 = np.array([15.0, 12.0, 15.0])

In [ ]:
answer_M25b_24a = ...  # a) "A", "B", "C" or "D": the unit and the order of the angles of subspace_angles (read its documentation)
angles_24 = answer_M25b_24b = answer_M25b_24c = answer_M25b_24d = ...
with wbm.attempt("M25b.24"):          # your sample_mvn, mvn_logpdf and condition_gaussian: ⏳ while they are not written
    angles_24 = ...        # the array that linalg.subspace_angles returns for the two planes (A_24, B_24), as returned
    answer_M25b_24b = ...  # b) True or False: np.cos of the angles (planes, then lines), ordered like your singular values of M25b.10 d), equals them to 1e-12
    answer_M25b_24c = ...  # c) a list of 6: σ̂², ‖ŵ‖², the mean log-likelihood per candidate (3 numbers, as computed), then 3 True or False: smaller than under N(x̄, Σ̂); lower with 1.1 σ̂²; larger than under N(μ, C)
    answer_M25b_24d = ...  # d) a list of 2 numbers, as computed: the posterior mean and variance of CANDIDATE_24 (condition_gaussian in dimension 4)

In [ ]:
wbm.check("M25b.24a", answer_M25b_24a)
with checked("M25b.24"):
    if filled(angles_24):
        want_c24 = linalg.subspace_angles(A_24, B_24)
        if matches(angles_24, np.degrees(want_c24)) or matches(angles_24, np.sort(want_c24)) or matches(
                angles_24, np.degrees(np.sort(want_c24))):
            print("❌ Ex M25b.24 : angles_24 doit être le tableau que rend subspace_angles, tel quel (sans conversion "
                  "ni tri).")
        else:
            same("M25b.24", "angles_24, les angles rendus pour les deux plans", angles_24, want_c24)
    else:
        print("⏳ Ex M25b.24 : la comparaison attend tes angles angles_24.")
for letter, answer in zip("bcd", [answer_M25b_24b, answer_M25b_24c, answer_M25b_24d]):
    wbm.check(f"M25b.24{letter}", answer, computed=True)

## Partie B · Prédire, mesurer, dessiner

Fiche §3, §4, §5 et §7. Une vraisemblance qui n'est pas bornée, prédite puis mesurée ; l'EM suivi demi-pas par demi-pas depuis plusieurs initialisations ; les deux sens de la KL, dessinés.

### Ex M25b.25 — Prédire ce que devient la log-vraisemblance quand une composante centrée sur un point se resserre (σ_k = 10⁻¹, 10⁻³, 10⁻⁶), puis avec une régularisation, et mesurer 🔮 ★ ⏱️ 15 min
**Objectif :** prévoir, puis mesurer, la croissance sans limite de la log-vraisemblance d'un mélange quand une composante s'effondre sur une donnée, et l'effet d'une régularisation.  
**Prérequis :** fiche §3 et §4 ; Ex M25b.3 et M25b.22 · **Parcours :** C

On reprend les quatre résistances de M25b.3 (`X_RES_25`) et le mélange de poids $(0{,}48 ;\ 0{,}52)$ (`WEIGHTS_25`) dont la composante 2 a pour moyenne 50,57 Ω et pour variance 2,36 Ω² (`MEAN2_25`, `VAR2_25`), et dont la composante 1 est **centrée sur la première mesure**, 46,0 Ω (`CENTRE1_25`), avec l'écart-type $\sigma_1$ (sa variance est $\sigma_1^2$). **Avant d'exécuter quoi que ce soit**, prédis :
- a) quand $\sigma_1$ passe de $10^{-1}$ à $10^{-3}$, la log-vraisemblance : (A) augmente d'environ 46 ; (B) ne change presque pas ; (C) augmente d'environ 4,6 ; (D) baisse (une lettre) ;
- b) quand $\sigma_1$ passe de $10^{-3}$ à $10^{-6}$, elle : (A) augmente d'environ 6,9 ; (B) augmente d'environ 4,6 ; (C) devient `inf` (une lettre) ;
- c) si l'on ajoute `reg = 0.01` à la variance de la composante 1, elle : (A) augmente des mêmes quantités ; (B) baisse à chaque fois ; (C) ne bouge presque plus dès $\sigma_1 = 10^{-3}$ (une lettre).

Puis, une fois tes prédictions écrites et vérifiées, mesure avec ta `gmm_log_likelihood`, pour $\sigma_1$ = $10^{-1}$, $10^{-3}$ et $10^{-6}$ (`SIGMAS_25`), sans puis avec `reg` (`REG_25`, ajouté à la variance de la seule composante 1), et relève :
- d) la log-vraisemblance pour $\sigma_1 = 10^{-6}$ sans régularisation (2 décimales dans ta copie) ;
- e) les trois valeurs avec régularisation (une liste de 3 nombres ; 3 décimales dans ta copie).

La vérification de l'expérience ne s'exécute qu'une fois tes trois prédictions écrites : elle compare tes trois log-vraisemblances sans régularisation (`lls_25`) à celles que calcule SciPy et affiche leurs incréments. Ce qu'il faut observer : chaque facteur 100 sur $\sigma_1$ ajoute environ $\ln 100$ nats, sans limite (une seule donnée expliquée par une aiguille) ; `reg` met un plancher sous la variance et borne la log-vraisemblance.

In [ ]:
X_RES_25 = np.array([[46.0], [48.0], [49.4], [52.1]])    # the four resistors of M25b.3 (Ω)
WEIGHTS_25 = np.array([0.48, 0.52])                        # the weights of the mixture
CENTRE1_25 = 46.0                                          # component 1: centred on the first measure, σ₁ varies
MEAN2_25, VAR2_25 = 50.57, 2.36                            # component 2: its mean (Ω) and its variance (Ω²)
SIGMAS_25 = [1e-1, 1e-3, 1e-6]                             # the standard deviations σ₁ of component 1
REG_25 = 0.01                                              # the regularization of c) and e)

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_M25b_25a = ...  # a) "A", "B", "C" or "D", written BEFORE the experiment: from σ₁ = 1e-1 to 1e-3, the log-likelihood...
prediction_M25b_25b = ...  # b) "A", "B" or "C": from σ₁ = 1e-3 to 1e-6, the log-likelihood...
prediction_M25b_25c = ...  # c) "A", "B" or "C": with reg = 0.01 added to the variance of component 1, the log-likelihood...

In [ ]:
for letter, prediction in zip("abc", [prediction_M25b_25a, prediction_M25b_25b, prediction_M25b_25c]):
    wbm.check(f"M25b.25{letter}", prediction)

**L'expérience.** Écris-la une fois tes trois prédictions vérifiées ; sa vérification ne s'exécute qu'à cette condition.

In [ ]:
lls_25 = answer_M25b_25d = answer_M25b_25e = ...
with wbm.attempt("M25b.25"):          # your gmm_log_likelihood: ⏳ while it is not written
    lls_25 = ...           # the three log-likelihoods without regularization, for σ₁ = 1e-1, 1e-3, 1e-6 (your gmm_log_likelihood)
    answer_M25b_25d = ...  # d) a number, as computed: the log-likelihood for σ₁ = 1e-6, without regularization
    answer_M25b_25e = ...  # e) a list of 3 numbers, as computed: the three log-likelihoods with REG_25 added to the variance of component 1

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_M25b_25a, prediction_M25b_25b, prediction_M25b_25c]):
    print("⏳ Écris d'abord tes trois prédictions, puis exécute à nouveau.")
else:
    with checked("M25b.25"):
        if not filled(lls_25):
            print("⏳ Ex M25b.25 : la vérification attend tes trois log-vraisemblances lls_25.")
            raise StopCheck

        def lls_c25(variance_of):
            """The three log-likelihoods by SciPy, the variance of component 1 given by variance_of(σ₁)."""
            return [scipy_log_likelihood(X_RES_25, WEIGHTS_25, np.array([[CENTRE1_25], [MEAN2_25]]),
                                         np.array([[[variance_of(s)]], [[VAR2_25]]])) for s in SIGMAS_25]

        want_c25 = lls_c25(lambda s: s ** 2)
        if matches(lls_25, lls_c25(lambda s: s), rtol=1e-9, atol=1e-9):
            print("❌ Ex M25b.25 : lls_25 met σ₁ à la place de la variance : covs contient σ₁².")
            raise StopCheck
        if same("M25b.25", "lls_25, les trois log-vraisemblances sans régularisation", lls_25, want_c25, rtol=1e-9, atol=1e-9):
            print(f"   incréments : {fr(want_c25[1] - want_c25[0])} et {fr(want_c25[2] - want_c25[1])} nats ; "
                  f"ln 100 ≈ {fr(math.log(100))}, ln 1000 ≈ {fr(math.log(1000))}")
    for letter, answer in zip("de", [answer_M25b_25d, answer_M25b_25e]):
        wbm.check(f"M25b.25{letter}", answer, computed=True)

**Après l'expérience**, dans ta copie : pourquoi chaque facteur 100 sur $\sigma_1$ ajoute-t-il presque exactement $\ln 100$ ? Quelle est la plus grande valeur possible du terme de 46,0 Ω avec `reg` ? Pourquoi une composante d'épaisseur nulle n'explique-t-elle rien, malgré sa vraisemblance infinie ?

### Ex M25b.26 — L'EM sur des longueurs de poissons de trois cohortes : la log-vraisemblance et l'ELBO après chaque demi-pas, sans régularisation, plusieurs initialisations et des maxima locaux 🔬 ★★ ⏱️ 25 min
**Objectif :** suivre l'EM demi-pas par demi-pas, voir l'ELBO monter et toucher la log-vraisemblance, et comparer plusieurs initialisations.  
**Prérequis :** fiche §4 et §5 ; Ex M25b.20 à M25b.22 · **Parcours :** C, E

Les longueurs (en cm) de poissons d'un étang viennent de trois cohortes, d'un, deux et trois ans : proportions $(0{,}5 ;\ 0{,}3 ;\ 0{,}2)$, moyennes 12, 20 et 28 cm, écarts-types 1,5 ; 2 et 2,5 cm. La cellule de données les tire par un tirage ancestral : `rng = np.random.default_rng(6056)`, `z = rng.choice(3, size=300, p=(0.5, 0.3, 0.2))`, puis, pour chaque cohorte $k = 0, 1, 2$, les longueurs des lignes où `z == k` par `sample_mvn([mu_k], [[var_k]], n_k, seed=6057 + k)` (celui de la référence) ; `X_FISH_26` est de forme $(300, 1)$.

Écris `em_trace_26(X, weights, means, covs, n_iter)`, qui fait `n_iter` itérations de l'EM sans régularisation avec tes fonctions `mymath.latent`, et rend deux tableaux `ll` et `elbo` de longueur `2 * n_iter + 1` : la log-vraisemblance et l'ELBO après le premier pas E (le posterior des paramètres de départ), puis après chaque pas M (les nouveaux paramètres, les mêmes responsabilités) et chaque pas E (les nouvelles responsabilités). Écris aussi `draw_trace_26(ax, ll, elbo, title)`, qui trace dans l'axe reçu `ll` et `elbo` contre le demi-pas $0, 1, \ldots$ (deux courbes), avec un titre et une légende.

Lance `em_trace_26` depuis dix initialisations : `rng = np.random.default_rng(6060)` puis, pour chacune, `idx = rng.choice(300, 3, replace=False)`, les moyennes `X[idx]`, trois variances égales à `X.var()`, des poids égaux, et `n_iter = 100` (`N_ITER_26`) ; `runs_26` est la liste des dix couples `(ll, elbo)`, dans l'ordre. Relève :
- a) vrai ou faux : dans les dix essais, aucun demi-pas ne fait baisser l'ELBO de plus de $10^{-9}(1 + \lvert\operatorname{ELBO}\rvert)$ ;
- b) vrai ou faux : après chaque pas E, l'ELBO et la log-vraisemblance diffèrent de moins de $10^{-9}(1 + \lvert\ell\rvert)$ ;
- c) la meilleure log-vraisemblance finale par poisson (4 décimales dans ta copie) ;
- d) le nombre d'essais dont la valeur finale par poisson est à moins de $10^{-3}$ de c) (un entier) ;
- e) la valeur finale par poisson des autres essais (elles sont égales à 2 décimales près ; donne l'une d'elles ; 2 décimales dans ta copie) ;
- f) les trois moyennes du meilleur essai, dans l'ordre croissant (une liste de 3 nombres ; 1 décimale dans ta copie).

La vérification essaie `em_trace_26` sur 3 itérations, compare tes dix essais à ceux de l'EM attendu, puis appelle `draw_trace_26` pour le meilleur essai et un des autres. Ce qu'il faut conclure : la log-vraisemblance monte en escalier et l'ELBO la touche après chaque pas E ; certaines initialisations restent longtemps près d'un moins bon point, où deux composantes se partagent les cohortes du haut : on relance depuis plusieurs initialisations.

In [ ]:
WEIGHTS_FISH_26 = (0.5, 0.3, 0.2)                           # three cohorts: one, two and three years old
MEANS_FISH_26 = (12.0, 20.0, 28.0)                         # their mean lengths (cm)
SDS_FISH_26 = (1.5, 2.0, 2.5)                              # their standard deviations (cm)
rng_fish_26 = np.random.default_rng(6056)                  # ancestral sampling: a cohort, then a length
Z_FISH_26 = rng_fish_26.choice(3, size=300, p=WEIGHTS_FISH_26)
X_FISH_26 = np.empty((300, 1))
for k_26 in range(3):
    rows_26 = np.flatnonzero(Z_FISH_26 == k_26)
    X_FISH_26[rows_26] = ref_gauss.sample_mvn([MEANS_FISH_26[k_26]], [[SDS_FISH_26[k_26] ** 2]], len(rows_26),
                                              seed=6057 + k_26)    # the reference sample_mvn: data never depend on your code
N_ITER_26 = 100

In [ ]:
def em_trace_26(X, weights, means, covs, n_iter):
    """n_iter iterations of EM without regularization (your mymath.latent); returns two arrays ll and elbo of length
    2 * n_iter + 1: the log-likelihood and the ELBO after the first E step (the posterior of the starting parameters),
    then after each M step and each E step."""
    raise NotImplementedError("em_trace_26 n'est pas encore écrite (04_indices.md si tu bloques)")


def draw_trace_26(ax, ll, elbo, title):
    """In ax: ll and elbo against the half step 0, 1, ..., len(ll) - 1 (two curves), the title, labels and a legend.
    No other axes."""
    raise NotImplementedError("draw_trace_26 n'est pas encore écrite (04_indices.md si tu bloques)")


runs_26 = answer_M25b_26a = answer_M25b_26b = answer_M25b_26c = answer_M25b_26d = answer_M25b_26e = answer_M25b_26f = ...
with wbm.attempt("M25b.26"):          # your em_trace_26 and your library: ⏳ while they are not written
    runs_26 = ...          # the ten pairs (ll, elbo) of em_trace_26, in the order of the initializations (default_rng(6060))
    answer_M25b_26a = ...  # a) True or False: in the ten runs, no half step lowers the ELBO by more than 1e-9 (1 + |ELBO|)
    answer_M25b_26b = ...  # b) True or False: after each E step, the ELBO and the log-likelihood differ by less than 1e-9 (1 + |ℓ|)
    answer_M25b_26c = ...  # c) a number, as computed: the best final log-likelihood per fish
    answer_M25b_26d = ...  # d) an integer: the number of runs whose final value per fish is within 1e-3 of c)
    answer_M25b_26e = ...  # e) a number, as computed: the final value per fish of the other runs (one of them)
    answer_M25b_26f = ...  # f) a list of 3 numbers, as computed: the three means of the best run, in increasing order

In [ ]:
for letter, answer in zip("abcdef", [answer_M25b_26a, answer_M25b_26b, answer_M25b_26c, answer_M25b_26d, answer_M25b_26e, answer_M25b_26f]):
    wbm.check(f"M25b.26{letter}", answer, computed=True)

with checked("M25b.26"):
    results_26 = []
    report_26 = reporter(results_26)
    if not filled(runs_26):
        print("⏳ Ex M25b.26 : la vérification attend tes dix essais runs_26.")
        raise StopCheck

    def trace_c26(X, weights, means, covs, n_iter):
        """The expected (ll, elbo) of EM without regularization, by the reference (the check never uses your code)."""
        resp = ref_latent.gmm_e_step(X, weights, means, covs)
        ll, elbo = [ref_latent.gmm_log_likelihood(X, weights, means, covs)], [ref_latent.gmm_elbo(X, resp, weights, means, covs)]
        for _ in range(n_iter):
            weights, means, covs = ref_latent.gmm_m_step(X, resp)
            ll.append(ref_latent.gmm_log_likelihood(X, weights, means, covs))
            elbo.append(ref_latent.gmm_elbo(X, resp, weights, means, covs))
            resp = ref_latent.gmm_e_step(X, weights, means, covs)
            ll.append(ll[-1])
            elbo.append(ref_latent.gmm_elbo(X, resp, weights, means, covs))
        return np.array(ll), np.array(elbo)

    def starts_c26(ddof=0):
        rng = np.random.default_rng(6060)
        return [(np.full(3, 1 / 3), X_FISH_26[rng.choice(300, 3, replace=False)],
                 np.full((3, 1, 1), X_FISH_26.var(ddof=ddof))) for _ in range(10)]

    first_c26 = starts_c26()[0]
    with own_code("M25b.26", "em_trace_26", REF_LATENT):
        short_c26 = em_trace_26(X_FISH_26.copy(), *(np.copy(v) for v in first_c26), 3)
    short_want_c26 = trace_c26(X_FISH_26, *first_c26, 3)
    got_c26 = pair("M25b.26", "em_trace_26", short_c26, names=("ll", "elbo"))
    if not (matches(got_c26[0], short_want_c26[0], rtol=1e-8) and matches(got_c26[1], short_want_c26[1], rtol=1e-8)):
        swapped_c26 = matches(got_c26[0], short_want_c26[1], rtol=1e-8) and matches(got_c26[1], short_want_c26[0], rtol=1e-8)
        report_26(False, "", "em_trace_26, 3 itérations depuis le premier départ : attendu ll = "
                  f"{shown(short_want_c26[0])} et elbo = {shown(short_want_c26[1])}, reçu {shown(got_c26[0])} et "
                  f"{shown(got_c26[1])}" + (" (ll et elbo échangés)" if swapped_c26 else " : 2 n_iter + 1 valeurs, "
                  "après le premier pas E, puis après chaque pas M et chaque pas E (et tes fonctions passent-elles "
                  "leurs tests ?)"))
    report_26(True, "em_trace_26 : 3 itérations depuis le premier départ, comme attendu", "")
    if not isinstance(runs_26, (list, tuple)) or len(runs_26) != 10:
        report_26(False, "", "runs_26 doit être une liste de 10 couples (ll, elbo), un par initialisation")
    want_c26 = [trace_c26(X_FISH_26, *start, N_ITER_26) for start in starts_c26()]
    try:
        ddof_c26 = trace_c26(X_FISH_26, *starts_c26(ddof=1)[0], N_ITER_26)
        if matches(runs_26[0][0], ddof_c26[0], rtol=1e-7):
            report_26(False, "", "tes variances de départ divisent par n − 1 : X.var() (np.var) divise par n ; ddof=1 "
                      "divise par n − 1")
    except (TypeError, IndexError, ValueError):
        pass
    for j_c26, (want_ll_c26, want_elbo_c26) in enumerate(want_c26):
        try:
            got_ll_c26, got_elbo_c26 = runs_26[j_c26]
        except (TypeError, ValueError):
            report_26(False, "", f"l'essai {j_c26 + 1} de runs_26 doit être un couple (ll, elbo)")
        ok_c26 = matches(got_ll_c26, want_ll_c26, rtol=1e-7) and matches(got_elbo_c26, want_elbo_c26, rtol=1e-7)
        if not ok_c26:
            same("M25b.26", f"l'essai {j_c26 + 1}, ll", got_ll_c26, want_ll_c26, rtol=1e-7)
            same("M25b.26", f"l'essai {j_c26 + 1}, elbo", got_elbo_c26, want_elbo_c26, rtol=1e-7)
            report_26(False, "", f"l'essai {j_c26 + 1} ne suit pas l'EM attendu : les départs se tirent dans l'ordre par "
                      "rng = np.random.default_rng(6060), rng.choice(300, 3, replace=False) pour chacun, trois "
                      "variances X.var(), des poids égaux, n_iter = 100")
    report_26(True, "les dix essais de runs_26 suivent l'EM attendu", "")
    finals_c26 = np.array([ll[-1] for ll, _ in want_c26]) / 300
    best_c26, worst_c26 = int(np.argmax(finals_c26)), int(np.argmin(finals_c26))
    fig_26, axes_26 = plt.subplots(1, 2, figsize=(12, 4))
    try:
        for ax_c26, j_c26, name_c26 in ((axes_26[0], best_c26, "le meilleur essai"), (axes_26[1], worst_c26, "un autre essai")):
            opened_c26 = drawn(draw_trace_26, ax_c26, *want_c26[j_c26], f"{name_c26} (essai {j_c26 + 1})")
            report_26(not opened_c26, f"{name_c26} : draw_trace_26 dessine dans l'axe reçu",
                      f"{name_c26} : draw_trace_26 doit dessiner dans l'axe reçu, sans ouvrir de figure")
            steps_c26 = np.arange(len(want_c26[j_c26][0]))
            report_26(through(ax_c26, steps_c26, want_c26[j_c26][0]) and through(ax_c26, steps_c26, want_c26[j_c26][1]),
                      f"{name_c26} : ll et elbo sont tracées contre le demi-pas",
                      f"{name_c26} : il manque une courbe, ll ou elbo, contre le demi-pas 0, 1, …, {len(steps_c26) - 1}")
    except StopCheck:
        fig_26.tight_layout()
        plt.show()
        raise
    except BaseException:
        plt.close(fig_26)
        raise
    fig_26.tight_layout()
    plt.show()
    verdict("M25b.26", all(results_26), "tes dix essais sont justes, et dessinés.", "")

**Après la figure**, dans ta copie : sur la courbe, où l'ELBO touche-t-elle la log-vraisemblance, et pourquoi l'escalier ne descend-il jamais ? Que font les deux composantes du haut dans les essais moins bons ? Combien d'initialisations faudrait-il pour être presque sûr de trouver le meilleur point ?

### Ex M25b.27 — Une gaussienne ajustée à un mélange bimodal par KL(p ‖ q) puis par KL(q ‖ p) : les deux courbes, et les deux minima de la KL inverse 🎨 ★ ⏱️ 20 min
**Objectif :** voir, sur un mélange bimodal, que la KL directe apparie les moments et que la KL inverse se pose sur une bosse.  
**Prérequis :** fiche §7 ; M25a ; Ex M25b.13 · **Parcours :** C

Écris `draw_kl_fits_27(ax_density, ax_kl, weights, means, variances)`, qui dessine sur les **axes reçus** (sans ouvrir de figure) :
- dans `ax_density`, sur $[0, 60]$ min (au moins 50 points), la densité $p$ du mélange (les temps de trajet de M25b.13 : `WEIGHTS_27`, `MEANS_27`, `VARIANCES_27`), la gaussienne $q_\star$ des moments, et les gaussiennes $q_1$ et $q_2$ des deux composantes, avec une légende ;
- dans `ax_kl`, pour $m$ de 10 à 55 par pas de 0,25 (`MS_27`), la KL inverse $m \mapsto \mathrm{KL}(\mathcal{N}(m, 4) \,\|\, p)$ (4 est la variance `variances[0]` de $q_1$) et la KL directe $m \mapsto \mathrm{KL}(p \,\|\, \mathcal{N}(m, \mathbb{V}_p(X)))$, calculées sur la grille de $x$ de $-30$ à 100 (260 001 points, `XS_KL_27`) par `np.trapezoid`, avec $\ln p$ par un log-sum-exp (les densités elles-mêmes sortiraient des flottants loin des bosses).

Appelle-la sur une figure de deux panneaux. Relève :
- a) les abscisses des minima locaux de la KL inverse sur la grille `MS_27` (une liste de 2 nombres ; 1 décimale dans ta copie) ;
- b) leurs valeurs (une liste de 2 nombres ; 3 décimales dans ta copie) ;
- c) $\mathrm{KL}(p \,\|\, q_\star)$ (4 décimales dans ta copie) ;
- d) $\mathrm{KL}(q_\star \,\|\, p)$ (2 décimales dans ta copie) ;
- e) vrai ou faux : $\mathrm{KL}(q_\star \,\|\, p)$ dépasse les deux minima de a).

Puis minimise $\mathrm{KL}(\mathcal{N}(m, s^2) \,\|\, p)$ en $(m, \ln s^2)$ par `scipy.optimize.minimize`, depuis $(22, \ln 4)$ et depuis $(40, \ln 6{,}25)$, et compare les deux minima à tes réponses c) et d) de M25b.13 (dans ta copie). La vérification appelle ta fonction pour les temps de trajet, puis pour un mélange que tu ne vois pas, et regarde chaque dessin. Ce qu'il faut observer : la KL inverse a un minimum par itinéraire, le plus bas sur la bosse la plus lourde ; la KL directe n'en a qu'un, entre les bosses, là où $p$ est presque nulle.

In [ ]:
WEIGHTS_27 = np.array([0.7, 0.3])                         # the travel times of M25b.13 (min): the ring road, the centre
MEANS_27 = np.array([22.0, 40.0])
VARIANCES_27 = np.array([4.0, 6.25])
MS_27 = np.arange(10.0, 55.0 + 1e-9, 0.25)                 # the means m of the two KL curves: 10, 10.25, ..., 55
XS_KL_27 = np.linspace(-30.0, 100.0, 260001)               # the grid of x of the integrals (np.trapezoid)

In [ ]:
def draw_kl_fits_27(ax_density, ax_kl, weights, means, variances):
    """In ax_density, on [0, 60] (at least 50 points): the density p of the mixture, the Gaussian q* of its moments, and
    the Gaussians q1 and q2 of its components, with a legend; in ax_kl, against MS_27: the reverse KL
    m ↦ KL(N(m, variances[0]) ‖ p) and the forward KL m ↦ KL(p ‖ N(m, V_p)), by np.trapezoid on XS_KL_27, ln p by a
    log-sum-exp; labels and a legend. No other axes."""
    raise NotImplementedError("draw_kl_fits_27 n'est pas encore écrite (04_indices.md si tu bloques)")


answer_M25b_27a = answer_M25b_27b = answer_M25b_27c = answer_M25b_27d = answer_M25b_27e = ...
with wbm.attempt("M25b.27"):          # ⏳ while your functions are not written
    answer_M25b_27a = ...  # a) a list of 2 numbers, as computed: the abscissae of the local minima of the reverse KL on MS_27
    answer_M25b_27b = ...  # b) a list of 2 numbers, as computed: their values
    answer_M25b_27c = ...  # c) a number, as computed: KL(p ‖ q*)
    answer_M25b_27d = ...  # d) a number, as computed: KL(q* ‖ p)
    answer_M25b_27e = ...  # e) True or False: KL(q* ‖ p) exceeds the two minima of a)

In [ ]:
for letter, answer in zip("abcde", [answer_M25b_27a, answer_M25b_27b, answer_M25b_27c, answer_M25b_27d, answer_M25b_27e]):
    wbm.check(f"M25b.27{letter}", answer, computed=True)

def curves_c27(weights, means, variances):
    """The expected curves on MS_27, by SciPy on the grid XS_KL_27: KL(N(m, variances[0]) ‖ p) and KL(p ‖ N(m, V_p))."""
    weights, means, variances = (np.asarray(v, dtype=float) for v in (weights, means, variances))
    log_p = special.logsumexp(np.log(weights) + stats.norm(means, np.sqrt(variances)).logpdf(XS_KL_27[:, None]), axis=1)
    mean_p = weights @ means
    var_p = weights @ variances + weights @ (means - mean_p) ** 2
    reverse, forward = [], []
    for m in MS_27:
        log_q = stats.norm(m, math.sqrt(variances[0])).logpdf(XS_KL_27)
        reverse.append(np.trapezoid(np.exp(log_q) * (log_q - log_p), XS_KL_27))
        forward.append(np.trapezoid(np.exp(log_p) * (log_p - stats.norm(m, math.sqrt(var_p)).logpdf(XS_KL_27)), XS_KL_27))
    return np.array(reverse), np.array(forward)


def look_27(ax_density, ax_kl, weights, means, variances):
    """What is wrong in a drawing of draw_kl_fits_27: a list of French phrases, empty when all is there."""
    weights, means, variances = (np.asarray(v, dtype=float) for v in (weights, means, variances))
    mean_p = weights @ means
    var_p = weights @ variances + weights @ (means - mean_p) ** 2
    densities = {"la densité p du mélange": lambda x: np.exp(special.logsumexp(
                     np.log(weights) + stats.norm(means, np.sqrt(variances)).logpdf(x[:, None]), axis=1)),
                 "la gaussienne q⋆ des moments": lambda x: stats.norm(mean_p, math.sqrt(var_p)).pdf(x),
                 "la gaussienne q₁": lambda x: stats.norm(means[0], math.sqrt(variances[0])).pdf(x),
                 "la gaussienne q₂": lambda x: stats.norm(means[1], math.sqrt(variances[1])).pdf(x)}
    missing = []
    for name, density in densities.items():
        if not any(line_points(line)[0].size >= 50 and line_points(line)[0].min() <= 1e-9
                   and line_points(line)[0].max() >= 60 - 1e-9
                   and np.allclose(line_points(line)[1], density(line_points(line)[0]), rtol=1e-6, atol=1e-12)
                   for line in ax_density.lines):
            missing.append(f"il manque {name} sur [0, 60] (au moins 50 points)")
    if ax_density.get_legend() is None:
        missing.append("il manque la légende du panneau des densités")
    reverse, forward = curves_c27(weights, means, variances)
    for name, curve in (("la KL inverse m ↦ KL(N(m, variances[0]) ‖ p)", reverse),
                        ("la KL directe m ↦ KL(p ‖ N(m, V_p))", forward)):
        if not through(ax_kl, MS_27, curve, rtol=1e-6, atol=1e-7):
            missing.append(f"il manque {name} contre MS_27 (np.trapezoid sur XS_KL_27, ln p par un log-sum-exp)")
    return missing


with checked("M25b.27"):
    results_27 = []
    report_27 = reporter(results_27)
    fig_27, (density_27, kl_27) = plt.subplots(1, 2, figsize=(13, 4.5))
    try:
        opened_27 = drawn(draw_kl_fits_27, density_27, kl_27, WEIGHTS_27.copy(), MEANS_27.copy(), VARIANCES_27.copy())
        report_27(not opened_27, "draw_kl_fits_27 dessine dans les axes reçus",
                  "draw_kl_fits_27 doit dessiner dans les deux axes reçus, sans ouvrir de figure")
        missing_27 = look_27(density_27, kl_27, WEIGHTS_27, MEANS_27, VARIANCES_27)
        report_27(not missing_27, "les quatre densités et les deux courbes de KL sont là", " ; ".join(missing_27))
        hidden_27 = (np.array([0.6, 0.4]), np.array([18.0, 37.0]), np.array([4.0, 6.25]))   # a mixture you do not see
        fig_c27, (density_c27, kl_c27) = plt.subplots(1, 2)
        try:
            opened_c27 = drawn(draw_kl_fits_27, density_c27, kl_c27, *(v.copy() for v in hidden_27))
            missing_c27 = look_27(density_c27, kl_c27, *hidden_27)
        finally:
            plt.close(fig_c27)
        report_27(not opened_c27 and not missing_c27, "un mélange que tu ne vois pas (d'autres poids, d'autres moyennes) "
                  "est dessiné juste aussi", "un mélange que tu ne vois pas (d'autres poids, d'autres moyennes) : "
                  + " ; ".join(missing_c27 or ["draw_kl_fits_27 doit dessiner dans les axes reçus, sans ouvrir de figure"]))
    except StopCheck:
        fig_27.tight_layout()
        plt.show()
        raise
    except BaseException:
        plt.close(fig_27)
        raise
    fig_27.tight_layout()
    plt.show()
    verdict("M25b.27", all(results_27), "les densités et les deux sens de la KL sont dessinés.", "")

**Après la figure**, dans ta copie : pourquoi la KL inverse a-t-elle deux minima et la KL directe un seul ? Où se trouve $q_\star$ par rapport aux bosses, et que vaut $p$ à cet endroit ? Laquelle des deux approximations choisirais-tu pour prévoir un temps de trajet typique, et laquelle pour ne jamais sous-estimer un retard ?

## Partie C · Déboguer, tester

Fiche §3, §4 et §5. Trois erreurs classiques des mélanges, qui passent une vérification mal choisie ; un test de propriété de l'EM. Les données et les vérifications s'appuient sur la référence (`ref_gauss`, `ref_latent`), pour ne pas dépendre de ton code.

### Ex M25b.28 — Chasse à l'erreur : des responsabilités sans log qui donnent 0/0, un pas M qui divise par n au lieu de N_k, et une ELBO sans l'entropie de q 🐛 ★ ⏱️ 20 min
**Objectif :** reconnaître trois erreurs classiques des mélanges qui passent une vérification mal choisie, puis les corriger.  
**Prérequis :** fiche §3, §4 et §5 ; Ex M25b.3, M25b.20 à M25b.22 · **Parcours :** C, E

Un collègue a écrit trois fonctions pour des mesures en dimension 1 (`x` est un tableau 1-D, `sds` les **écarts-types**), chacune accompagnée d'une vérification qui passe (cellule fournie) :
1. `machine_shares(x, weights, means, sds)` doit rendre les responsabilités, de forme $(n, K)$ ; elle calcule chaque $\pi_k\,\mathcal{N}(x_i \mid \mu_k, \sigma_k^2)$ par `np.exp`, puis divise chaque ligne par sa somme. Il l'a vérifiée sur les quatre résistances de M25b.3 (paramètres de départ, `sds` = les racines de 1 et 2,5) : chaque ligne a pour somme 1 et toutes les valeurs sont dans $[0, 1]$. Il l'applique au bac qui contient aussi la résistance de 120 Ω (`X_BIN_28`).
2. `refit_machines(x, resp)` doit rendre le pas M, `(weights, means, variances)` ; elle calcule les poids $N_k/n$, les moyennes $\frac{1}{N_k}\sum_ir_{ik}x_i$, puis les variances $\frac1n\sum_ir_{ik}(x_i - \mu_k)^2$. Il l'a vérifiée avec une seule composante (une colonne de 1) : elle rend bien `np.mean(x)` et `np.var(x)`.
3. `bound_of_evidence(x, resp, weights, means, sds)` doit rendre l'ELBO ; elle rend $\sum_i\sum_kr_{ik}\big(\ln\pi_k + \ln\mathcal{N}(x_i \mid \mu_k, \sigma_k^2)\big)$. Il l'a vérifiée sur 200 tableaux de responsabilités tirés au hasard (`np.random.default_rng(6066).dirichlet`) : elle reste toujours sous la log-vraisemblance des quatre résistances.

À faire : trouve et corrige chaque erreur ; les fonctions corrigées s'appellent `machine_shares_fixed`, `refit_machines_fixed` et `bound_of_evidence_fixed`, avec les mêmes arguments (sans SciPy, et sans jamais calculer $0/0$ ni $\ln 0$). Puis réponds :
- a) vrai ou faux : la dernière ligne que rend `machine_shares` pour les cinq résistances contient des `nan` ;
- b) cette ligne, par `machine_shares_fixed` (une liste de 2 nombres ; 4 décimales dans ta copie) ;
- c) les variances que rend `refit_machines` pour les quatre résistances, avec les responsabilités de `machine_shares_fixed` (une liste de 2 nombres ; 3 décimales dans ta copie) ;
- d) `bound_of_evidence` au posterior (les responsabilités de `machine_shares_fixed`) moins la log-vraisemblance, pour les quatre résistances (4 décimales dans ta copie).

Pour chacune des trois erreurs, dis aussi, dans ta copie, pourquoi la vérification du collègue ne la voyait pas. La vérification essaie tes corrections sur des cas que tu ne vois pas : des mélanges de 2 à 4 composantes, des mesures très loin de toutes les composantes, des responsabilités quelconques, et des lignes qui contiennent des zéros.

In [ ]:
X_RES_28 = np.array([46.0, 48.0, 49.4, 52.1])             # the four resistors of M25b.3 (Ω): a 1-D array here
X_BIN_28 = np.append(X_RES_28, 120.0)                      # the bin, with the stray resistor of 120 Ω
WEIGHTS_28, MEANS_28 = np.array([0.45, 0.55]), np.array([47.0, 51.0])
SDS_28 = np.sqrt(np.array([1.0, 2.5]))                     # the STANDARD DEVIATIONS of the two series


# The colleague's code
def machine_shares(x, weights, means, sds):
    """The responsibilities (n, K) of a mixture of normal laws in dimension 1 (x 1-D, sds the standard deviations)."""
    x = np.asarray(x, dtype=float)[:, None]
    dens = weights * np.exp(-(x - means) ** 2 / (2 * sds ** 2)) / (sds * np.sqrt(2 * np.pi))
    return dens / dens.sum(axis=1, keepdims=True)


def refit_machines(x, resp):
    """The M step in dimension 1: (weights, means, variances), each of shape (K,)."""
    x, resp = np.asarray(x, dtype=float)[:, None], np.asarray(resp, dtype=float)
    n, N = len(x), resp.sum(axis=0)
    means = (resp * x).sum(axis=0) / N
    return N / n, means, (resp * (x - means) ** 2).sum(axis=0) / n


def bound_of_evidence(x, resp, weights, means, sds):
    """The ELBO of the mixture for the laws q_i given by the rows of resp."""
    x, resp = np.asarray(x, dtype=float)[:, None], np.asarray(resp, dtype=float)
    log_dens = -np.log(sds * np.sqrt(2 * np.pi)) - (x - means) ** 2 / (2 * sds ** 2)
    return float(np.sum(resp * (np.log(weights) + log_dens)))


# The colleague's checks: all pass
R_check_28 = machine_shares(X_RES_28, WEIGHTS_28, MEANS_28, SDS_28)
print("(1) quatre résistances : lignes de somme 1 :", bool(np.allclose(R_check_28.sum(axis=1), 1)),
      "; valeurs dans [0, 1] :", bool(np.all((R_check_28 >= 0) & (R_check_28 <= 1))), "(attendu True, True)")
one_28 = refit_machines(X_RES_28, np.ones((4, 1)))
print("(2) une seule composante : np.mean :", bool(np.isclose(one_28[1][0], np.mean(X_RES_28))), "; np.var :",
      bool(np.isclose(one_28[2][0], np.var(X_RES_28))), "(attendu True, True)")
dens_check_28 = WEIGHTS_28 * np.exp(-(X_RES_28[:, None] - MEANS_28) ** 2 / (2 * SDS_28 ** 2)) / (SDS_28 * np.sqrt(2 * np.pi))
loglik_check_28 = float(np.sum(np.log(dens_check_28.sum(axis=1))))     # the log-likelihood of the four resistors
rng_check_28 = np.random.default_rng(6066)
below_28 = all(bound_of_evidence(X_RES_28, rng_check_28.dirichlet(np.ones(2), size=4), WEIGHTS_28, MEANS_28, SDS_28)
               <= loglik_check_28 for _ in range(200))
print("(3) 200 tableaux tirés au hasard : ELBO sous la log-vraisemblance :", below_28, "(attendu True)")

In [ ]:
def machine_shares_fixed(x, weights, means, sds):
    """The responsibilities (n, K) of a mixture of normal laws in dimension 1 (x 1-D, sds the standard deviations)."""
    raise NotImplementedError("machine_shares_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


def refit_machines_fixed(x, resp):
    """The M step in dimension 1: (weights, means, variances), each of shape (K,)."""
    raise NotImplementedError("refit_machines_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


def bound_of_evidence_fixed(x, resp, weights, means, sds):
    """The ELBO of the mixture for the laws q_i given by the rows of resp (0 ln 0 counts for 0): a float."""
    raise NotImplementedError("bound_of_evidence_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


answer_M25b_28a = answer_M25b_28b = answer_M25b_28c = answer_M25b_28d = ...
with wbm.attempt("M25b.28"):          # your three functions: ⏳ while they are not written
    answer_M25b_28a = ...  # a) True or False: the last row that the WRONG machine_shares returns for X_BIN_28 holds nan
    answer_M25b_28b = ...  # b) a list of 2 numbers, as computed: that row by machine_shares_fixed
    answer_M25b_28c = ...  # c) a list of 2 numbers, as computed: the variances of the WRONG refit_machines for X_RES_28, with the responsibilities of machine_shares_fixed
    answer_M25b_28d = ...  # d) a number, as computed: the WRONG bound_of_evidence at the posterior minus the log-likelihood, for X_RES_28

In [ ]:
for letter, answer in zip("abcd", [answer_M25b_28a, answer_M25b_28b, answer_M25b_28c, answer_M25b_28d]):
    wbm.check(f"M25b.28{letter}", answer, computed=True)

results_28, parts_28 = [], []


def report_28(ok, success, failure):
    results_28.append(bool(ok))
    print(("✅ " + success) if ok else ("❌ " + failure))


def call_28(part, name, function, *args):
    """function(*args); an error of the learner's code (a RuntimeWarning included: 0/0, ln 0) becomes one ❌ line
    that names the function (the check of this part stops there, the final verdict stays)."""
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("error", RuntimeWarning)
            return function(*args)
    except (NotImplementedError, MymathMissingError):
        raise
    except Exception as exc:  # noqa: BLE001 - the learner's error, reported with the name of the function
        hint = " (un 0/0 ou un ln 0 : calcule en log, et compte 0 ln 0 pour 0)" if isinstance(exc, RuntimeWarning) else ""
        report_28(False, "", f"({part}) {name} : ton code lève {type(exc).__name__} : {str(exc)[:200]}{hint}")
        parts_28.append(part)
        raise StopCheck from None


rng_c28 = np.random.default_rng(6068)


def law_c28(K):
    """Random weights, means and standard deviations of a mixture in dimension 1."""
    return rng_c28.dirichlet(2 * np.ones(K)), rng_c28.normal(0, 5, K), rng_c28.uniform(0.5, 2.0, K)


with checked("M25b.28"):                                      # (1) machine_shares_fixed
    wrong_28 = []
    for K_c28, n_c28 in ((2, 6), (3, 7), (4, 5)):
        w_c28, m_c28, s_c28 = law_c28(K_c28)
        x_c28 = np.concatenate([rng_c28.normal(0, 4, n_c28 - 2), [m_c28.max() + 300, m_c28.min() - 250]])
        want_c28 = special.softmax(np.log(w_c28) + stats.norm(m_c28, s_c28).logpdf(x_c28[:, None]), axis=1)
        got_c28 = call_28(1, "machine_shares_fixed", machine_shares_fixed, x_c28.copy(), w_c28.copy(), m_c28.copy(),
                          s_c28.copy())
        if not matches(got_c28, want_c28):
            wrong_28.append(f"{K_c28} composantes, des mesures lointaines : attendu un tableau {want_c28.shape} de "
                            f"dernière ligne {np.round(want_c28[-1], 4).tolist()}, reçu {shown(got_c28)[:120]}")
    report_28(not wrong_28, "(1) machine_shares_fixed : des mélanges de 2 à 4 composantes, des mesures très loin de "
              "toutes les composantes", "(1) machine_shares_fixed, " + " ; ".join(wrong_28[:2]))
    parts_28.append(1)

with checked("M25b.28"):                                      # (2) refit_machines_fixed
    wrong_28 = []
    for K_c28, n_c28 in ((2, 8), (3, 9)):
        x_c28 = rng_c28.normal(10, 3, n_c28)
        resp_c28 = rng_c28.dirichlet(np.ones(K_c28), size=n_c28)
        N_c28 = resp_c28.sum(axis=0)
        want_c28 = (N_c28 / n_c28, np.array([np.average(x_c28, weights=resp_c28[:, k]) for k in range(K_c28)]),
                    np.array([np.cov(x_c28, aweights=resp_c28[:, k], bias=True) for k in range(K_c28)]))
        got_c28 = call_28(2, "refit_machines_fixed", refit_machines_fixed, x_c28.copy(), resp_c28.copy())
        if not isinstance(got_c28, (tuple, list)) or len(got_c28) != 3:
            wrong_28.append(f"attendu le triplet (weights, means, variances), reçu {shown(got_c28)[:90]}")
            break
        if all(matches(g, w) for g, w in zip(got_c28, want_c28)):
            continue
        if matches(got_c28[2], want_c28[2] * N_c28 / (N_c28 - 1)):
            wrong_28.append("tes variances divisent par N_k − 1 : le pas M divise par N_k (maximum de vraisemblance)")
        elif matches(got_c28[2], want_c28[2] * N_c28 / n_c28):
            wrong_28.append("tes variances divisent encore par n")
        else:
            wrong_28.append(f"{K_c28} composantes, des responsabilités quelconques : attendu "
                            f"{shown(tuple(want_c28))[:160]}, reçu {shown(tuple(got_c28))[:160]}")
    report_28(not wrong_28, "(2) refit_machines_fixed : 2 et 3 composantes, des responsabilités quelconques",
              "(2) refit_machines_fixed : " + " ; ".join(list(dict.fromkeys(wrong_28))[:2]))
    parts_28.append(2)

with checked("M25b.28"):                                      # (3) bound_of_evidence_fixed
    wrong_28 = []
    for K_c28, n_c28 in ((2, 5), (3, 6)):
        w_c28, m_c28, s_c28 = law_c28(K_c28)
        x_c28 = rng_c28.normal(0, 4, n_c28)
        resp_c28 = rng_c28.dirichlet(np.ones(K_c28), size=n_c28)
        resp_c28[0] = np.eye(K_c28)[0]                        # rows with zeros: 0 ln 0 counts for 0
        resp_c28[1] = np.eye(K_c28)[-1]
        a_c28 = np.log(w_c28) + stats.norm(m_c28, s_c28).logpdf(x_c28[:, None])
        want_c28 = float(np.sum(resp_c28 * a_c28) + np.sum(special.entr(resp_c28)))
        got_c28 = call_28(3, "bound_of_evidence_fixed", bound_of_evidence_fixed, x_c28.copy(), resp_c28.copy(),
                          w_c28.copy(), m_c28.copy(), s_c28.copy())
        try:
            got_value_c28 = float(got_c28)
        except (TypeError, ValueError):
            wrong_28.append(f"attendu un nombre, reçu {shown(got_c28)[:90]}")
            break
        if math.isclose(got_value_c28, want_c28, rel_tol=1e-9, abs_tol=1e-12):
            continue
        if math.isclose(got_value_c28, float(np.sum(resp_c28 * a_c28)), rel_tol=1e-9, abs_tol=1e-12):
            wrong_28.append("il manque encore l'entropie de q")
        elif math.isclose(got_value_c28, float(np.sum(resp_c28 * a_c28) - np.sum(special.entr(resp_c28))), rel_tol=1e-9):
            wrong_28.append("l'entropie de q est comptée avec le mauvais signe : l'ELBO retranche E_q[ln q]")
        else:
            wrong_28.append(f"{K_c28} composantes, des lignes avec des zéros : attendu {want_c28:.6f}, reçu "
                            f"{got_value_c28:.6f}")
    report_28(not wrong_28, "(3) bound_of_evidence_fixed : 2 et 3 composantes, des lignes avec des zéros",
              "(3) bound_of_evidence_fixed : " + " ; ".join(list(dict.fromkeys(wrong_28))[:2]))
    parts_28.append(3)

if len(set(parts_28)) == 3:
    verdict("M25b.28", all(results_28), "les trois fonctions corrigées passent tous les cas.",
            "relis les lignes ❌ ci-dessus.")

**Après la vérification**, dans ta copie : pour chaque erreur, pourquoi la vérification du collègue ne la voyait-elle pas (des mesures proches des séries, une seule composante, une inégalité au lieu de l'égalité au posterior), et quelle vérification l'aurait vue ? Pour (3), pourquoi l'erreur garde-t-elle l'inégalité ELBO ≤ ℓ ?

### Ex M25b.29 — Un test de propriété : sur des mélanges tirés au hasard, ELBO ≤ log-vraisemblance, égalité après le pas E, et un EM sans régularisation (reg = 0) qui ne descend jamais (à une tolérance près) 🛠️ ★ ⏱️ 15 min
**Objectif :** écrire un test de propriété qui compte ses échecs sur beaucoup de cas tirés au hasard, avec une tolérance justifiée, et qui échoue sur un pas faux.  
**Prérequis :** fiche §4 et §5 ; Ex M25b.20 à M25b.22 et M25b.26 · **Parcours :** C

Écris `em_property_test_29(X, weights, means, covs, m_step, n_iter, tol, rng)`, qui fait `n_iter` itérations de l'EM depuis les paramètres donnés (le pas E de ton `gmm_e_step`, puis le pas M `m_step(X, resp)` reçu) et rend un dictionnaire de trois compteurs (des nombres entiers) :
- `"bound"`, le nombre d'itérations où l'ELBO d'un tableau de responsabilités tiré au hasard (`rng.dirichlet(np.ones(K), size=n)`, un par itération) dépasse la log-vraisemblance de plus de `tol * (1 + |ℓ|)` ;
- `"gap"`, le nombre d'itérations où l'ELBO au posterior (après le pas E) s'écarte de la log-vraisemblance de plus de `tol * (1 + |ℓ|)` ;
- `"descent"`, le nombre de demi-pas où l'ELBO baisse de plus de `tol * (1 + |ELBO|)`.

La cellule fournie donne `random_mixture_29(rng)`, qui tire un mélange (dimension de 1 à 5, de 2 à 4 composantes), 200 données de ce mélange et des paramètres de départ (le quadruplet `(X, weights, means, covs)`), et `m_step_means_over_n(X, resp)`, un pas M faux qui divise les moyennes par $n$.
- a) Dans ta copie : justifie `tol = 1e-9` : de quel ordre sont les erreurs d'arrondi sur une somme de quelques centaines de termes ?
- b) Sur 40 mélanges (`rng = np.random.default_rng(6061)`, puis `random_mixture_29(rng)` pour chacun, et ce même `rng` pour ton test), avec `gmm_m_step` sans régularisation et `n_iter = 30` : vrai ou faux, les trois compteurs restent nuls (un mélange où `gmm_e_step` lève une `ValueError`, une composante effondrée, se compte à part et se saute).
- c) Vrai ou faux : sur les mêmes 40 mélanges (un nouveau `default_rng(6061)`), avec `m_step_means_over_n`, ton test compte au moins une descente.
- d) Vrai ou faux : de même, avec `lambda X, r: gmm_m_step(X, r, reg=0.1)`, ton test compte au moins une descente.

La vérification lance ton test sur deux mélanges que tu ne vois pas : avec le pas M de la référence (`ref_latent`), où ses compteurs doivent rester nuls, puis avec des pas M faux (celui de l'énoncé, un autre que tu ne vois pas, et `reg = 0.1`), où il doit compter des descentes ; un entier NumPy convient pour un compteur. Ce qu'il faut retenir : un test de propriété compte ses échecs sur beaucoup de cas tirés au hasard, avec une tolérance justifiée, et doit échouer sur un pas faux ; avec `reg > 0`, la monotonie n'est plus garantie.

In [ ]:
def random_mixture_29(rng):
    """A random mixture (dimension 1 to 5, 2 to 4 components), 200 data drawn from it (ancestral sampling, the reference
    sample_mvn), and starting parameters: (X, weights, means, covs) with equal weights, the means on K distinct data
    and every covariance equal to the empirical covariance of X (bias=True)."""
    d, K, n = int(rng.integers(1, 6)), int(rng.integers(2, 5)), 200
    true_weights, true_means = rng.dirichlet(3 * np.ones(K)), rng.normal(0, 3, (K, d))
    A = rng.normal(size=(K, d, d))
    true_covs = A @ A.transpose(0, 2, 1) / d + 0.5 * np.eye(d)
    z = rng.choice(K, size=n, p=true_weights)
    X = np.empty((n, d))
    for k in range(K):
        rows = np.flatnonzero(z == k)
        if len(rows):
            X[rows] = ref_gauss.sample_mvn(true_means[k], true_covs[k], len(rows), seed=int(rng.integers(2**31)))
    start_cov = np.cov(X.T, bias=True).reshape(d, d)
    return X, np.ones(K) / K, X[rng.choice(n, K, replace=False)], np.tile(start_cov, (K, 1, 1))


def m_step_means_over_n(X, resp):
    """A WRONG M step: the means divided by n instead of N_k (the weights N_k / n, the covariances around these means,
    divided by N_k)."""
    X, resp = np.asarray(X, dtype=float), np.asarray(resp, dtype=float)
    n, N = len(X), resp.sum(axis=0)
    means = resp.T @ X / n
    covs = np.array([(resp[:, k, None] * (X - means[k])).T @ (X - means[k]) / N[k] for k in range(len(N))])
    return N / n, means, covs

In [ ]:
def em_property_test_29(X, weights, means, covs, m_step, n_iter, tol, rng):
    """n_iter iterations of EM from the given parameters: the E step of your gmm_e_step, then the M step m_step(X, resp).
    Returns {"bound": ..., "gap": ..., "descent": ...}: the number of iterations where the ELBO of a random table
    rng.dirichlet(np.ones(K), size=n) exceeds the log-likelihood by more than tol * (1 + |ℓ|); the number of iterations
    where the ELBO at the posterior differs from the log-likelihood by more than tol * (1 + |ℓ|); the number of half steps
    where the ELBO goes down by more than tol * (1 + |ELBO|)."""
    raise NotImplementedError("em_property_test_29 n'est pas encore écrite (04_indices.md si tu bloques)")


answer_M25b_29b = answer_M25b_29c = answer_M25b_29d = ...
with wbm.attempt("M25b.29"):          # your test and your library: ⏳ while they are not written
    answer_M25b_29b = ...  # b) True or False: on 40 mixtures (rng = np.random.default_rng(6061)), gmm_m_step, n_iter = 30, tol = 1e-9: the three counters stay at 0 (a mixture where gmm_e_step raises ValueError is skipped)
    answer_M25b_29c = ...  # c) True or False: with m_step_means_over_n, your test counts at least one descent
    answer_M25b_29d = ...  # d) True or False: with lambda X, r: gmm_m_step(X, r, reg=0.1), your test counts at least one descent

In [ ]:
for letter, answer in zip("bcd", [answer_M25b_29b, answer_M25b_29c, answer_M25b_29d]):
    wbm.check(f"M25b.29{letter}", answer, computed=True)

with checked("M25b.29"):
    results_29 = []
    report_29 = reporter(results_29, stop=False)

    def case_c29(rng, weights, means, covs, start_at_truth=False, n=200):
        """A mixture you do not see: 200 data by ancestral sampling, and starting parameters."""
        weights, means, covs = (np.asarray(v, dtype=float) for v in (weights, means, covs))
        z = rng.choice(len(weights), size=n, p=weights)
        X = np.empty((n, means.shape[1]))
        for k in range(len(weights)):
            rows = np.flatnonzero(z == k)
            X[rows] = ref_gauss.sample_mvn(means[k], covs[k], len(rows), seed=int(rng.integers(2**31)))
        starts = rng.choice(n, len(weights), replace=False)
        if start_at_truth:
            return X, weights, means, covs
        return X, np.ones(len(weights)) / len(weights), X[starts], np.tile(np.cov(X.T, bias=True), (len(weights), 1, 1))

    rng_c29 = np.random.default_rng(6069)
    cases_c29 = [case_c29(rng_c29, [0.3, 0.3, 0.4], [[4.0, 1.0], [7.5, 5.0], [2.0, 6.5]],
                          [[[0.06, 0.02], [0.02, 0.05]], [[0.08, -0.02], [-0.02, 0.06]], [[0.05, 0.0], [0.0, 0.09]]],
                          start_at_truth=True),
                 case_c29(rng_c29, [0.55, 0.45], [[5.0, 0.5, 3.0], [1.0, 4.0, 6.0]],
                          [np.diag([0.3, 0.2, 0.25]) + 0.05, np.diag([0.2, 0.35, 0.15]) + 0.03])]

    def totals_29(m_step, label):
        """The counters of em_property_test_29 summed over the two hidden mixtures (30 iterations, tol = 1e-9)."""
        totals = {"bound": 0, "gap": 0, "descent": 0}
        for case in cases_c29:
            try:
                counts = em_property_test_29(*(np.copy(v) for v in case), m_step, 30, 1e-9, np.random.default_rng(6081))
            except (NotImplementedError, MymathMissingError):
                raise
            except AssertionError as exc:
                print(f"❌ Ex M25b.29 : em_property_test_29 ({label}) lève AssertionError ({str(exc)[:120]}) : il doit "
                      "compter les échecs et rendre le dictionnaire, sans assert.")
                raise StopCheck from None
            except Exception as exc:  # noqa: BLE001 - the learner's error
                print(f"❌ Ex M25b.29 : em_property_test_29 ({label}) lève {type(exc).__name__} : {str(exc)[:200]}")
                raise StopCheck from None
            if not isinstance(counts, dict) or set(counts) != set(totals):
                print(f"❌ Ex M25b.29 : em_property_test_29 doit rendre un dictionnaire de clés \"bound\", \"gap\" et "
                      f"\"descent\", reçu {shown(counts)[:120]}")
                raise StopCheck
            for key, value in counts.items():
                if isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, np.integer)):
                    print(f"❌ Ex M25b.29 : le compteur \"{key}\" doit être un nombre entier de cas, reçu "
                          f"{type(value).__name__} {value!r}")
                    raise StopCheck
                totals[key] += int(value)
        return totals

    totals_c29 = totals_29(ref_latent.gmm_m_step, "pas M de la référence")
    report_29(all(v == 0 for v in totals_c29.values()), "avec le pas M de la référence, sur deux mélanges que tu ne vois "
              "pas, tes trois compteurs restent nuls", f"avec le pas M de la référence, tes compteurs valent "
              f"{totals_c29} au lieu de 0 : vérifie ton test (les tolérances relatives, l'ordre des demi-pas), puis tes "
              "fonctions de M25b.20 à M25b.22 (leurs tests)")
    wrong_c29 = (("le pas M faux de l'énoncé (m_step_means_over_n)", m_step_means_over_n),
                 ("un pas M faux que tu ne vois pas (des covariances deux fois trop grandes)",
                  lambda X, r: tuple(v * f for v, f in zip(ref_latent.gmm_m_step(X, r), (1.0, 1.0, 2.0)))),
                 ("le pas M de la référence avec reg = 0.1", lambda X, r: ref_latent.gmm_m_step(X, r, reg=0.1)))
    for label_c29, step_c29 in wrong_c29:
        found_c29 = totals_29(step_c29, label_c29)
        report_29(found_c29["descent"] >= 1, f"ton test compte des descentes avec {label_c29} ({found_c29['descent']})",
                  f"ton test ne compte aucune descente avec {label_c29} : compare l'ELBO après chaque demi-pas à "
                  "celle d'avant")
    try:
        mine_c29 = totals_29(mymath.latent.gmm_m_step, "ta librairie")
        print("✅ ta librairie (gmm_m_step) passe ton test" if all(v == 0 for v in mine_c29.values()) else
              f"❌ ta librairie : avec ton gmm_m_step, tes compteurs valent {mine_c29} (lance aussi ses tests, M25b.21)")
    except StopCheck:
        pass
    verdict("M25b.29", all(results_29), "ton test reste muet sur la référence et attrape les pas M faux.",
            "relis les lignes ❌ ci-dessus.")

**Dans ta copie** : a) la justification de `tol = 1e-9`.

**Après la vérification**, dans ta copie : pourquoi `m_step_means_over_n` et `reg = 0.1` font-ils descendre l'ELBO, alors que le pas E ne la fait jamais baisser ? Que testerait en plus un test qui partirait d'un tableau `q` quelconque, avec un pas M d'abord ?

## Partie D · Défi

Fiche §3, §4 et §5. Trois régimes d'une rivière, retrouvés par l'EM et jugés sur des mesures de test.

### Ex M25b.30 — Défi : retrouver trois populations de mesures en dimension 2 par l'EM, sous un seuil de log-vraisemblance moyenne sur des mesures de test 🏆 ★★★ ⏱️ 25 min
**Objectif :** écrire un EM complet (initialisations, régularisation, arrêt, meilleur essai) avec ta librairie, et le juger sur des données de test.  
**Prérequis :** fiche §3, §4 et §5 ; M25a §3 ; Ex M25b.20 à M25b.22 et M25b.26 · **Parcours :** C

Une sonde mesure dans une rivière la turbidité (en NTU) et la conductivité (en centaines de µS/cm) ; l'eau est dans l'un de trois régimes, étiage, normal ou crue, que la sonde ne voit pas. La cellule de données tire 600 mesures d'entraînement (`X_TRAIN_30`) et 400 mesures de test (`X_TEST_30`) par un tirage ancestral (`draw_regimes_30(n, seed)`, graines 6062 et 6070, avec le `sample_mvn` de la référence), d'un mélange de trois gaussiennes dont elle garde les paramètres pour elle.

Écris `fit_gmm_30(X, K, n_init, n_iter, reg, seed)`, qui rend `(weights, means, covs)` à partir de tes fonctions `mymath.latent` : `n_init` initialisations (`rng = np.random.default_rng(seed)`, puis, pour chacune, les moyennes sur $K$ mesures distinctes `X[rng.choice(n, K, replace=False)]`, chaque covariance égale à la covariance globale `np.cov(X, rowvar=False, bias=True)` plus `reg` fois l'identité, des poids égaux), au plus `n_iter` itérations de l'EM avec la régularisation `reg`, un arrêt quand la log-vraisemblance bouge de moins de $10^{-8}\lvert\ell\rvert$, et l'essai de meilleure log-vraisemblance d'entraînement. Écris aussi `draw_fit_30(ax, X, weights, means, covs)`, qui trace dans l'axe reçu les mesures, colorées par la composante de plus grande responsabilité, et l'ellipse à 90 % de chaque composante (l'image du cercle de rayon $c$, $c^2 = 2\ln 10$, par $\boldsymbol{\varepsilon} \mapsto \boldsymbol{\mu}_k + \mathbf{L}_k\boldsymbol{\varepsilon}$, au moins 50 points ; M25a §3).

**Critère de réussite :** la vérification appelle `fit_gmm_30(X_TRAIN_30, 3, **SETTINGS_30)` (`n_init=5`, `n_iter=300`, `reg=1e-6`, `seed=6080`) et réussit si la log-vraisemblance moyenne des mesures de test, `gmm_log_likelihood(X_TEST_30, …) / 400`, est au moins $-3{,}38$ (`THRESHOLD_30`). Elle compare ensuite ton ajustement au modèle exact : après la meilleure renumérotation des composantes, des poids à 0,05 près, des moyennes à 0,3 près et des covariances à 0,4 près, coordonnée par coordonnée, et une log-vraisemblance de test à 0,02 près de celle du modèle exact ; puis elle ajuste un mélange que tu ne vois pas, et regarde ta figure. Relève, dans ta copie : a) ta log-vraisemblance moyenne de test (affichée par la vérification) ; b) celle du modèle exact (affichée aussi) ; c) celles de tes ajustements à $K = 2$ et $K = 6$ (affichées aussi).

Pour aller plus loin : trace la log-vraisemblance moyenne d'entraînement et de test en fonction de $K$, de 1 à 6 ; où l'écart entre les deux grandit-il ?

In [ ]:
def draw_regimes_30(n, seed):
    """n measures (turbidity in NTU, conductivity in hundreds of µS/cm) of the river, by ancestral sampling from a mixture
    of three Gaussians (low water, normal, flood) whose parameters stay inside this function: a regime z by
    np.random.default_rng(seed), then a measure of its normal law (the reference sample_mvn, seed + 1 + k)."""
    weights = np.array([0.5, 0.3, 0.2])
    means = np.array([[1.5, 6.5], [4.0, 4.5], [9.0, 2.5]])
    covs = np.array([[[0.3, 0.1], [0.1, 0.5]], [[1.0, -0.3], [-0.3, 0.6]], [[3.0, -0.8], [-0.8, 0.8]]])
    z = np.random.default_rng(seed).choice(3, size=n, p=weights)
    X = np.empty((n, 2))
    for k in range(3):
        rows = np.flatnonzero(z == k)
        X[rows] = ref_gauss.sample_mvn(means[k], covs[k], len(rows), seed=seed + 1 + k)
    return X


X_TRAIN_30 = draw_regimes_30(600, 6062)                    # the training measures
X_TEST_30 = draw_regimes_30(400, 6070)                     # the test measures
SETTINGS_30 = {"n_init": 5, "n_iter": 300, "reg": 1e-6, "seed": 6080}   # the arguments of the check
THRESHOLD_30 = -3.38                                       # the criterion: mean log-likelihood of the test measures

In [ ]:
def fit_gmm_30(X, K, n_init, n_iter, reg, seed):
    """(weights, means, covs) of a mixture of K Gaussians fitted to X by EM with your mymath.latent: n_init
    initializations (rng = np.random.default_rng(seed); for each, the means on K distinct measures
    X[rng.choice(n, K, replace=False)], each covariance the global covariance of X (bias=True) plus reg times the
    identity, equal weights), at most n_iter iterations of EM with the regularization reg, stopped when the
    log-likelihood moves by less than 1e-8 |ℓ|; returns the initialization of best training log-likelihood."""
    raise NotImplementedError("fit_gmm_30 n'est pas encore écrite (04_indices.md si tu bloques)")


def draw_fit_30(ax, X, weights, means, covs):
    """In ax: the measures X (a scatter), coloured by the component of largest responsibility, and the 90 % ellipse of
    each component (the image of the circle of radius c, c² = 2 ln 10, by ε ↦ μ_k + L_k ε, at least 50 points);
    labels and a legend. No other axes."""
    raise NotImplementedError("draw_fit_30 n'est pas encore écrite (04_indices.md si tu bloques)")

In [ ]:
fitted_30 = None
with checked("M25b.30"):
    TRUE_C30 = (np.array([0.5, 0.3, 0.2]), np.array([[1.5, 6.5], [4.0, 4.5], [9.0, 2.5]]),
                np.array([[[0.3, 0.1], [0.1, 0.5]], [[1.0, -0.3], [-0.3, 0.6]], [[3.0, -0.8], [-0.8, 0.8]]]))

    def valid_c30(result, K, name="fit_gmm_30"):
        """The fit as three float arrays of the right shapes, finite, of weights summing to 1 and SPD covariances; else
        one ❌ line and StopCheck."""
        w, m, C = pair("M25b.30", name, result, names=("weights", "means", "covs"))
        try:
            w, m, C = (np.asarray(v, dtype=float) for v in (w, m, C))
        except (TypeError, ValueError):
            verdict("M25b.30", False, "", f"{name} doit rendre trois tableaux de nombres (weights, means, covs).")
            raise StopCheck from None
        if w.shape != (K,) or m.shape != (K, 2) or C.shape != (K, 2, 2) or not all(np.all(np.isfinite(v)) for v in (w, m, C)):
            verdict("M25b.30", False, "", f"{name} doit rendre des tableaux finis de formes ({K},), ({K}, 2) et ({K}, 2, 2) ; "
                    f"reçu {w.shape}, {m.shape} et {C.shape}.")
            raise StopCheck
        if np.any(w <= 0) or abs(w.sum() - 1) > 1e-9 or not all(np.all(np.linalg.eigvalsh((c + c.T) / 2) > 0) for c in C):
            verdict("M25b.30", False, "", f"{name} : des poids positifs de somme 1 et des covariances définies positives, "
                    "s'il te plaît.")
            raise StopCheck
        return w, m, C

    def matched_c30(fit, truth):
        """The fit with its components renumbered like the truth (the permutation of the closest means), and the gaps."""
        import itertools
        best = min(itertools.permutations(range(3)), key=lambda p: np.abs(fit[1][list(p)] - truth[1]).max())
        w, m, C = (v[list(best)] for v in fit)
        return (w, m, C), (np.abs(w - truth[0]).max(), np.abs(m - truth[1]).max(), np.abs(C - truth[2]).max())

    with own_code("M25b.30", "fit_gmm_30", REF_LATENT):
        result_c30 = fit_gmm_30(X_TRAIN_30.copy(), 3, **SETTINGS_30)
    fit_c30 = valid_c30(result_c30, 3)
    test_c30 = scipy_log_likelihood(X_TEST_30, *fit_c30) / len(X_TEST_30)
    exact_c30 = scipy_log_likelihood(X_TEST_30, *TRUE_C30) / len(X_TEST_30)
    print(f"a) ta log-vraisemblance moyenne de test : {fr(test_c30, 4)} ; b) celle du modèle exact : {fr(exact_c30, 4)}")
    passed_c30 = test_c30 >= THRESHOLD_30
    verdict("M25b.30", passed_c30, f"critère atteint : {fr(test_c30, 4)} ≥ {fr(THRESHOLD_30, 2)}.",
            f"critère pas encore atteint : {fr(test_c30, 4)} < {fr(THRESHOLD_30, 2)}.")
    (w_c30, m_c30, C_c30), (dw_c30, dm_c30, dC_c30) = matched_c30(fit_c30, TRUE_C30)
    dll_c30 = abs(test_c30 - exact_c30)
    lead_c30 = "le critère est atteint, mais " if passed_c30 else "de plus, "
    if dw_c30 <= 0.05 and dm_c30 <= 0.3 and dC_c30 <= 0.4 and dll_c30 <= 0.02:
        print(f"✅ Ex M25b.30 : ton ajustement est celui du modèle exact, aux fluctuations de l'échantillon près (poids à "
              f"{fr(dw_c30, 3)}, moyennes à {fr(dm_c30, 2)}, covariances à {fr(dC_c30, 2)}, log-vraisemblance de test à "
              f"{fr(dll_c30, 4)}).")
    else:
        reasons_c30 = []                                      # first the classic constructions, then the gaps
        if np.allclose(w_c30, 1 / 3, rtol=0, atol=1e-9):
            reasons_c30.append("tes poids sont restés égaux : le pas M met π_k = N_k / n")
        if np.allclose(C_c30[:, 0, 1], 0, rtol=0, atol=1e-12):
            reasons_c30.append("tes covariances sont diagonales : le pas M garde toute la covariance pondérée, termes hors "
                               "diagonale compris")
        ratio_c30 = np.trace(C_c30, axis1=1, axis2=2) / np.trace(TRUE_C30[2], axis1=1, axis2=2)
        if np.all(ratio_c30 < 0.75) and dC_c30 > 0.4:
            reasons_c30.append("tes covariances sont trop petites : le pas M divise par N_k, pas par n")
        gaps_c30 = (f"écarts au modèle : poids {fr(dw_c30, 3)}, moyennes {fr(dm_c30, 2)}, covariances {fr(dC_c30, 2)}, "
                    f"log-vraisemblance de test {fr(dll_c30, 4)} (tolérances 0,05, 0,3, 0,4 et 0,02)")
        if reasons_c30:
            reasons_c30.append(gaps_c30)
        specific_c30 = bool(reasons_c30)                      # a classic construction found: the gaps in one line
        if not specific_c30 and dw_c30 > 0.05:
            reasons_c30.append(f"tes poids s'écartent de ceux du modèle de {fr(dw_c30, 3)} (plus de 0,05)")
        if not specific_c30 and dm_c30 > 0.3:
            reasons_c30.append(f"tes moyennes s'écartent de celles du modèle de {fr(dm_c30, 2)} (plus de 0,3) : un moins bon "
                               "maximum local ? garde l'essai de meilleure log-vraisemblance d'entraînement")
        if not specific_c30 and dC_c30 > 0.4:
            reasons_c30.append(f"tes covariances s'écartent de celles du modèle de {fr(dC_c30, 2)} (plus de 0,4)")
        if not specific_c30 and dll_c30 > 0.02:
            reasons_c30.append(f"ta log-vraisemblance de test s'écarte de {fr(dll_c30, 4)} de celle du modèle exact (plus "
                               "de 0,02)")
        print(f"❌ Ex M25b.30 : {lead_c30}ton ajustement n'est pas celui du modèle exact, après la meilleure "
              "renumérotation des composantes : " + " ; ".join(dict.fromkeys(reasons_c30)) + ".")
    for K_c30 in (2, 6):
        with own_code("M25b.30", "fit_gmm_30", REF_LATENT):
            other_c30 = valid_c30(fit_gmm_30(X_TRAIN_30.copy(), K_c30, **SETTINGS_30), K_c30)
        print(f"c) à K = {K_c30} : log-vraisemblance moyenne de test {fr(scipy_log_likelihood(X_TEST_30, *other_c30) / 400, 4)}")
    fitted_30 = fit_c30

with checked("M25b.30"):
    hidden_c30 = (np.array([0.35, 0.4, 0.25]), np.array([[2.0, 3.0], [6.0, 7.5], [8.5, 3.5]]),
                  np.array([[[0.5, 0.2], [0.2, 0.4]], [[0.8, -0.2], [-0.2, 0.5]], [[0.6, 0.0], [0.0, 1.2]]]))
    z_c30 = np.random.default_rng(6075).choice(3, size=600, p=hidden_c30[0])
    X_hidden_c30 = np.empty((600, 2))
    for k_c30 in range(3):
        rows_c30 = np.flatnonzero(z_c30 == k_c30)
        X_hidden_c30[rows_c30] = ref_gauss.sample_mvn(hidden_c30[1][k_c30], hidden_c30[2][k_c30], len(rows_c30),
                                                      seed=6076 + k_c30)
    with own_code("M25b.30", "fit_gmm_30", REF_LATENT):
        hidden_fit_c30 = valid_c30(fit_gmm_30(X_hidden_c30, 3, **SETTINGS_30), 3)
    _, (dw_c30, dm_c30, dC_c30) = matched_c30(hidden_fit_c30, hidden_c30)
    verdict("M25b.30", dw_c30 <= 0.05 and dm_c30 <= 0.3 and dC_c30 <= 0.4,
            "un mélange que tu ne vois pas (trois autres régimes) est retrouvé aussi.",
            f"un mélange que tu ne vois pas (trois autres régimes) n'est pas retrouvé : poids à {fr(dw_c30, 3)}, moyennes à "
            f"{fr(dm_c30, 2)}, covariances à {fr(dC_c30, 2)} (tolérances 0,05, 0,3 et 0,4).")

with checked("M25b.30"):
    if fitted_30 is None:
        print("⏳ Ex M25b.30 : la figure attend un ajustement qui passe la vérification ci-dessus.")
        raise StopCheck
    fig_30, ax_30 = plt.subplots(figsize=(7.5, 5.5))
    try:
        opened_30 = drawn(draw_fit_30, ax_30, X_TRAIN_30.copy(), *(np.copy(v) for v in fitted_30))
    except BaseException:
        plt.close(fig_30)
        raise
    missing_30 = [] if has_points(ax_30, X_TRAIN_30) else ["il manque des mesures : un point par ligne de X_TRAIN_30"]
    missing_30 += [f"il manque l'ellipse à 90 % de la composante {k + 1} (au moins 50 points, Δ² = 2 ln 10)"
                   for k in range(3) if not ellipse_drawn(ax_30, fitted_30[1][k], fitted_30[2][k], 0.9)]
    if opened_30:
        missing_30.insert(0, "draw_fit_30 doit dessiner dans l'axe reçu, sans ouvrir de figure")
    plt.show()
    verdict("M25b.30", not missing_30, "les mesures et les trois ellipses à 90 % sont dessinées.", " ; ".join(missing_30))

**Dans ta copie** : a), b) et c), lus sur la vérification ; pourquoi $K = 2$ passe-t-il sous le seuil, et pourquoi $K = 6$ fait-il mieux sur l'entraînement mais moins bien sur le test ?

**Pour réfléchir** : pourquoi une petite régularisation (`reg = 1e-6`) suffit-elle ici, alors que la vraisemblance d'un mélange n'est pas bornée ? Que se passerait-il si une initialisation plaçait deux moyennes sur la même mesure ?

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu écrire la loi jointe d'un modèle à variable latente, en tirer un échantillon ancestral, et l'inverser par la règle de Bayes ?
2. Sais-tu calculer les responsabilités et la log-vraisemblance d'un mélange en log, sans jamais obtenir $0/0$, et dire pourquoi cette vraisemblance n'est pas bornée ?
3. Sais-tu écrire le pas M (diviser par $N_k$, la nouvelle moyenne) et expliquer pourquoi l'EM sans régularisation ne descend jamais ?
4. Sais-tu calculer une ELBO pour une loi $q$ quelconque, et lire son écart à la log-vraisemblance comme une somme de KL ?
5. Sais-tu ajuster une ACP probabiliste par ses valeurs propres, comparer deux sous-espaces par leurs angles principaux, et choisir le sens de la KL ?

**La suite** : M26, qui rappelle l'espérance conditionnelle et en fait un outil, puis le chapitre 25 de workbookIA, qui construit des modèles génératifs à variables latentes, écrit leur ELBO « reconstruction moins KL au prior », et dérive à travers des tirages.